# cells

> The SolveIt-style cell GUI: a stack of typed cells (code / note / prompt / raw),
each toggleable in/out of the LLM's view. Because the model call is stateless,
context is just re-assembled from whichever cells are currently visible.

Note cells are markdown (KaTeX math, images, raw HTML). Standard Jupyter
command-mode hotkeys drive selection and editing.

In [ ]:
#| default_exp cells

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import os, shutil, subprocess, sys, threading, time, json, re, signal, secrets
from pathlib import Path
from fastcore.utils import *
from fasthtml.common import *
from fasthtml.jupyter import *
import fasthtml.components as fh
from fasthtml.svg import Path as SvgPath  # fastcore's pathlib.Path shadows fasthtml's svg <path> element otherwise
from fasthtml.svg import G as SvgG  # groups a sub-path so it can be scaled independently of its parent icon
from datetime import datetime
from lisette import *
from boopiter.notebook import *  # CTYPES, Cell, Notebook, and the single `nb` instance -- a mutable singleton imported by reference and only ever mutated in place, never rebound
from boopiter.kernel import run_code, _shell, _RunState, _run_code_bg, _collapse_cr  # the execution engine; the underscore names aren't in kernel.__all__ so they're imported explicitly
from boopiter.serialize import *  # save_notebook / load_notebook -- the .ipynb file boundary
from boopiter.llms import *  # get_tool_list, get_model_list, prompt_llm -- generic LLM utilities with no dependency on this module's Notebook/Cell
from boopiter.llms import _reply_details_html, _PREFERRED_MODEL_SUBSTR  # underscore-prefixed -- not in llms.py's __all__, so import * won't bring them in
from boopiter.plugins import *  # PLUGINS/start_plugins/Plugin -- a leaf module (imports only external libs, never this one), so no circular import; see nbs/03_plugins.ipynb

## DaisyUI + app setup

CDN headers for DaisyUI + Tailwind, plus `KatexMarkdownJS()` (renders `.marked`
elements as markdown + KaTeX) and the command-mode hotkey listener.

In [ ]:
#| export
# Define the path to the static directory
try:                                    # normal cli run
    static_dir = Path(__file__).parent / 'static'
except NameError:                       # notebook / nbdev-test
    from nbdev.config import get_config
    static_dir = get_config().lib_path / 'static'
    
# Function to read a file and return its content as a string
def read_file_content(file_path):
    with open(file_path, 'r') as file:
        return file.read()

# Read each JavaScript file and assign its content to the corresponding variable
_HOTKEYS_JS  = read_file_content(static_dir / 'hotkeys.js')
_MARKED_CSS  = read_file_content(static_dir / 'marked.css')
_EDIT_JS     = read_file_content(static_dir / 'edit.js')
_THEME_JS    = read_file_content(static_dir / 'theme.js')
_MARKDOWN_JS = read_file_content(static_dir / 'markdown.js')


In [ ]:
#| export
def _tw_header():
    "Use the precompiled static Tailwind build (from `_build_tailwind()`) if it exists; else fall back to the slower CDN JIT compiler. `__file__` isn't defined when nbdev-test executes this notebook directly (vs. a real module import), so fall back to cwd -- the .exists() check below fails safely either way."
    pkg_dir = Path(__file__).parent if '__file__' in globals() else Path.cwd()
    compiled = pkg_dir/'static/tailwind.css'
    if compiled.exists(): return Link(rel='stylesheet', href='/tailwind.css')
    return Script(src='https://cdn.jsdelivr.net/npm/@tailwindcss/browser@4')

In [ ]:
#| export
daisy_hdrs = [
    Link(href='https://cdn.jsdelivr.net/npm/daisyui@5', rel='stylesheet', type='text/css'),
    _tw_header(),
    Link(rel='stylesheet', href='https://cdn.jsdelivr.net/npm/katex@0.16.11/dist/katex.min.css'),
    Script(_MARKDOWN_JS, type='module'),
    Link(id='hljs-theme', rel='stylesheet',
         href='https://cdn.jsdelivr.net/gh/highlightjs/cdn-release/build/styles/atom-one-dark.min.css'),
    Script(src='https://cdn.jsdelivr.net/gh/highlightjs/cdn-release/build/highlight.min.js'),
    Script(src='https://cdn.jsdelivr.net/gh/highlightjs/cdn-release/build/languages/python.min.js'),
    Link(rel='stylesheet', href='https://cdn.jsdelivr.net/npm/codemirror@5/lib/codemirror.min.css'),
    Link(rel='stylesheet', href='https://cdn.jsdelivr.net/npm/codemirror@5/theme/material-darker.min.css'),
    Script(src='https://cdn.jsdelivr.net/npm/codemirror@5/lib/codemirror.min.js'),
    Script(src='https://cdn.jsdelivr.net/npm/codemirror@5/mode/python/python.min.js'),
    Script(src='https://cdn.jsdelivr.net/npm/codemirror@5/addon/comment/comment.min.js'),
    Link(rel='icon', type='image/png', href='/logo.png'),
    Style(_MARKED_CSS),
    # DaisyUI's default tooltip font-size (.875rem) renders like a native OS tooltip -- bump it so
    # hover hints are actually legible instead of squinting-required. .tooltip-right-align keeps a
    # bottom-tooltip but anchors its right edge (instead of centering) to the target, for elements
    # flush against the viewport's right edge where a centered tooltip would get clipped.
    Style('.tooltip[data-tip]:before{font-size:.88rem}'
          '.tooltip-right-align[data-tip]:before{left:auto;right:0;transform:translateY(var(--tt-pos,-.25rem))}'),
    # .icon-fillable always renders fill='none' as a plain SVG attribute; .icon-filled overrides it
    # via CSS (which wins over a presentation attribute), so JS can flip an icon between outline and
    # solid instantly (classList.toggle) with zero network round-trip -- see boopToggleExport() in
    # edit.js, used for the export-flag bookmark icon so clicking it doesn't need a whole-cell
    # outerHTML swap (and the flicker that came with one) just to change one icon's fill.
    Style('.icon-fillable{fill:none}.icon-fillable.icon-filled{fill:currentColor}'),
    Script("import { AnsiUp } from 'https://cdn.jsdelivr.net/npm/ansi_up@6/ansi_up.js';"
           " window.AnsiUp = AnsiUp; if(window.boopRenderAnsi) boopRenderAnsi();", type='module'),
    Script(_THEME_JS),
    Script(_HOTKEYS_JS),
    Script(_EDIT_JS),
]


## Authentication

A `?token=` query param (printed at startup) or the `/login` form gates every route except health-check/login itself -- see `_check_auth`, the `Beforeware` wired into `app` below. Every code cell already runs with this process's own OS permissions, so an unauthenticated boopiter listening on a shared network is equivalent to an open shell; `boopiter launch --no_auth` opts out explicitly for a fully trusted network.

In [ ]:
#| export
_AUTH_SKIP = [r'/_boopiter_ping', r'/login']  # paths that must work before a session is authed

def _login_page(error:str=None) -> FT:
    "Standalone token-entry page. Deliberately has zero dependency on the app's own static assets (no CDN links, no /tailwind.css, no /logo.png) -- so it always renders even if something upstream of auth is broken, and doesn't widen the auth skip list beyond /login itself."
    return Html(
        Head(Title('boopiter -- sign in'),
             Style('body{background:#1d232a;color:#ecf9ff;font-family:system-ui,sans-serif;'
                   'display:flex;align-items:center;justify-content:center;height:100vh;margin:0}'
                   '.box{background:#191e24;padding:2rem;border-radius:.5rem;width:22rem}'
                   'input{width:100%;padding:.5rem;margin:.75rem 0;border-radius:.25rem;border:none;'
                   'background:#282a36;color:#ecf9ff;box-sizing:border-box;font-size:1rem}'
                   'button{width:100%;padding:.5rem;border-radius:.25rem;border:none;'
                   'background:#605dff;color:white;cursor:pointer;font-size:1rem}'
                   '.err{color:#ff627d}')),
        Body(Div(H2('boopiter'),
                 P("Enter the access token printed in the server's console output at startup."),
                 P(error, cls='err') if error else '',
                 Form(Input(type='password', name='token', placeholder='token', autofocus=True),
                      Button('Sign in', type='submit'),
                      method='post', action='/login'),
                 cls='box')))

def _check_auth(req, session):
    "Beforeware (wired into `app = FastHTML(..., before=...)` below): gates every route except _AUTH_SKIP behind BOOPITER_TOKEN, Jupyter-style. A `?token=` query param matching the real token stamps the session (signed cookie) once and redirects to the clean URL -- the cookie then carries every later request. No match returns the login page, not a bare 403, so a missing/expired session is recoverable without restarting the server. Set BOOPITER_NO_AUTH=1 (`boopiter launch --no_auth`) to disable entirely -- only on a network you fully trust, since every code cell already runs with this process's own permissions (see kernel.run_code); with auth off, anyone who can reach the port has that."
    if os.environ.get('BOOPITER_NO_AUTH') == '1': return None
    if session.get('boop_authed'): return None
    real = os.environ.get('BOOPITER_TOKEN', '')
    qtoken = req.query_params.get('token')
    if real and qtoken and secrets.compare_digest(qtoken, real):
        session['boop_authed'] = True
        return RedirectResponse(req.url.path, status_code=303)
    return _login_page()


In [ ]:
#| export
app = FastHTML(hdrs=daisy_hdrs, htmlkw={'data-theme':'dark'}, before=Beforeware(_check_auth, skip=_AUTH_SKIP))
rt  = app.route
p   = partial(HTMX, app=app, host=None, port=None)


In [ ]:
#| export
@rt('/login')
def login(token:str=None, session=None):
    "Token-gate entry point. GET (no `token` field submitted) shows the sign-in form; POST checks it against BOOPITER_TOKEN and, on match, stamps the session so _check_auth lets every other route through. The other way in is a `?token=` query param on any URL -- what the server prints at startup -- this form is the fallback for typing the token in by hand."
    if token is None: return _login_page()
    real = os.environ.get('BOOPITER_TOKEN', '')
    if real and secrets.compare_digest(token, real):
        session['boop_authed'] = True
        return RedirectResponse('/', status_code=303)
    return _login_page(error='Incorrect token.')


## Code execution

A single IPython shell backs every code cell (like the lesson's `ex`), with errors
returned as text instead of raised.

In [ ]:
#| export
_run_state:'_RunState|None' = None  # the one code cell currently executing in the background, if any
_run_all_queue:list[int] = []  # remaining code-cell ids for an in-flight Run All (drained by run_code_poll as each cell finishes)
_run_all_current:'int|None' = None  # the cell id Run All is currently on; only its completion advances the queue, so a solo run's completion can never hijack a lingering batch

In [ ]:
#| export
_POLL_SCHEDULE = (0, 40, 80, 150, 300)  # ms delay before each successive still-running poll; holds at the last value thereafter. Most cells finish before the first (zero-delay) poll even fires -- by the time an HTTP response reaches the browser and htmx reissues a request, a trivial cell's background thread has typically already completed. Only genuinely slow cells fall back to something close to the original fixed 300ms cadence, so a long-running cell never gets hammered.

def _poll_trigger(poll_n:int) -> str:
    "htmx hx-trigger value for the `poll_n`-th poll of a streaming placeholder -- see _POLL_SCHEDULE."
    delay = _POLL_SCHEDULE[min(poll_n, len(_POLL_SCHEDULE) - 1)]
    return 'load' if delay == 0 else f'load delay:{delay}ms'

In [ ]:
#| export
def _run_output_div(id:int, text:str, poll_n:int=0) -> FT:
    "The small, self-polling output area shown under a code cell while its execution is still running. Only this div re-swaps on each poll tick (not the whole cell -- see pending_code_cell()), so the static code block above it doesn't flicker. `poll_n` (0 for the very first poll, right after the cell started) escalates the delay before each successive re-poll -- see _POLL_SCHEDULE -- so a cell that's already finished by the time this placeholder reaches the browser resolves on the very next round trip instead of waiting out a fixed delay."
    content = [Pre(_collapse_cr(text), cls='ansi-out text-sm mt-1 whitespace-pre overflow-x-auto opacity-70')] if text else []
    return Div(*content, id=f'run-out-{id}', hx_post=run_code_poll.to(id=id, poll_n=poll_n+1),
               hx_target=f'#run-out-{id}', hx_swap='outerHTML', hx_trigger=_poll_trigger(poll_n))

In [ ]:
#| export
@rt
def run_code_poll(id:int, poll_n:int=0) -> FT|tuple:
    "Poll a running code cell's execution. While still running, returns just the small output div (re-triggering itself, with an escalating delay -- see _POLL_SCHEDULE) so the code block above it never flickers. Once done, replaces the whole cell out-of-band -- the primary target (#run-out-N) is about to be destroyed along with it, so the primary response body is empty."
    global _run_state, _run_all_queue, _run_all_current
    c = nb.get(id)
    if not c: return ''
    st = _run_state
    if st is None or st.cell_id != id:
        return '', render_cell(c, oob=True)  # stale poll (e.g. a second tab, or after an interrupt already finalized it) -- just resync
    if not st.done:
        return _run_output_div(id, ''.join(st.buffer), poll_n=poll_n)
    c.output = st.blocks
    _run_state = None
    if id == _run_all_current:  # only the cell Run All is currently on advances the batch -- a solo play/Shift-Enter completion never does, even if a stale queue somehow lingers
        _run_all_current = None
        if any(b['type'] == 'error' for b in st.blocks):
            _run_all_queue = []  # stop-on-error: abandon the rest of the Run All sequence
        else:
            while _run_all_queue:  # skip any queued cells that were deleted mid-run
                nxt = nb.get(_run_all_queue.pop(0))
                if nxt:
                    _run_all_current = nxt.id
                    return '', render_cell(c, oob=True), _oob(f'outerHTML:#cell-{nxt.id}', run_code_cell(nxt, scroll=True))
    return '', render_cell(c, oob=True)

## Cell + Notebook model

A `Cell` carries its type, source, optional output, and a `visible` flag (the eye
toggle — whether the LLM sees it). `Notebook` is the in-memory store of cells, the
composer's selected type, and the currently `selected` cell (for hotkeys).

In [ ]:
#| export
def pending_code_cell(c:'Cell', text:str='', scroll:bool=False) -> FT:
    "Placeholder for a code cell whose execution is still running in the background: a static code view plus a self-polling output area (_run_output_div) that swaps itself out for the real render_cell() once done. `scroll=True` (set only by Run All) tags the cell div so the client scrolls it into view as it starts -- see boopScrollRunAll() in edit.js."
    attrs = {'data_runall_scroll': '1'} if scroll else {}
    return _cell_outer(c, cell_header(c, running=True),
                        Pre(Code(c.source, cls='language-python'), cls='text-sm overflow-x-auto'),
                        _run_output_div(c.id, text), **attrs)

In [ ]:
#| export
def _start_code_run(c:'Cell', scroll:bool=False) -> FT:
    "Kick off `c.source` running in a background thread and return a placeholder that polls for its progress; see run_code_poll(). `scroll` is forwarded to the placeholder -- see run_code_cell/pending_code_cell."
    global _run_state
    if nb.selected == c.id: nb.selected = None  # revert to the static (fast) view immediately, like every other cell type
    state = _RunState(c.id)
    state.thread = threading.Thread(target=_run_code_bg, args=(c.source, state), daemon=True)
    _run_state = state
    state.thread.start()
    return pending_code_cell(c, scroll=scroll)

In [ ]:
#| export
def run_code_cell(c:'Cell', scroll:bool=False) -> FT:
    "Tell one code cell to run: stamp it with the current time, then kick off its streaming background execution. The single shared entry point for the play button, Shift-Enter, and Run All (which passes scroll=True so each cell scrolls into view as it starts). A solo run (scroll=False) cancels any in-flight or leftover Run All, so it can never chain into unrelated cells."
    global _run_all_queue, _run_all_current
    if not scroll: _run_all_queue, _run_all_current = [], None
    c.ts = datetime.now().strftime('%I:%M:%S %p')
    return _start_code_run(c, scroll=scroll)

In [ ]:
#| export
BROWSE_ROOT = Path.cwd()  # file browser is rooted here (wherever `boopiter` was launched from), like Jupyter

## LLM Interaction

The whole point of the visibility toggle: context is only the *visible* cells. The
stub proves the plumbing by reporting what it can see; swap `stub_reply` for a real
model call later.

In [ ]:
#| export
_prompt_state:'_RunState|None' = None  # the one Prompt cell currently streaming a reply, if any -- parallel to _run_state (code), not shared with it: a code cell and a prompt reply use different resources and can run at the same time.

def _run_prompt_bg(context:str, model:str, tools:list, think:str|None, state:_RunState) -> None:
    "Runs in a background thread: drives llms.stream_llm_reply() (which owns the actual model-calling strategy, including how tools are exposed and reasoning effort is applied -- see there) into state.buffer/state.blocks. This function's only job is the background-thread/UI-state plumbing shared with code-cell streaming; it has no LLM-specific logic of its own."
    try:
        for item in stream_llm_reply(context, model, tools, think):
            if item[0] == 'final':
                _, content, details = item
                state.blocks = (content, details)
            else:
                _, delta = item
                state.buffer.append(delta)
    except KeyboardInterrupt:
        state.blocks = (''.join(state.buffer) + '\n\n*(interrupted)*', None)
    except Exception as e:
        state.blocks = (f'(error calling {model}: {e})', None)
    state.done = True

In [ ]:
#| export
def _start_prompt_run(prompt_id:int) -> None:
    "Kick off the LLM call for `prompt_id`, streaming into _prompt_state -- in a background thread if a real model is configured, or resolved immediately via stub_reply() if not (no need for a thread when there's no real call to make). Uses nb.active_model() (standard or reasoning, per the brain-icon toggle) and, while reasoning is active, nb.reasoning_effort."
    global _prompt_state
    c = nb.get(prompt_id)
    state = _RunState(prompt_id)
    model = nb.active_model()
    if model:
        tools = get_tool_list(nb.tool_selection) + nb.tools
        _push_tools()  # keep the shell namespace in sync before the model can suggest calling any of these
        think = nb.reasoning_effort if nb.use_reasoning else None
        state.thread = threading.Thread(target=_run_prompt_bg, args=(llm_context(nb, prompt_id), model, tools, think, state), daemon=True)
        _prompt_state = state
        state.thread.start()
    else:
        state.blocks = (stub_reply(nb, c.source), None)
        state.done = True
        _prompt_state = state

In [ ]:
#| export
def pending_prompt_cell(prompt_id:int, text:str='', oob_swap:str=None) -> FT:
    "Placeholder shown while a Prompt's LLM reply streams in the background: a pulsing 'Tricky...' indicator plus whatever text has accumulated so far, shown as plain preformatted text (not markdown-rendered -- mid-stream markdown is often invalid, e.g. an unclosed code fence; only the finished reply gets the full '.marked' treatment). hx-trigger=load polls run_prompt_poll() every 300ms until the reply is complete. `oob_swap`, if given, delivers this placeholder out-of-band (see _oob()) instead of being the response's main swap target."
    kw = {'hx_swap_oob': oob_swap} if oob_swap else {}
    parts = [Span('Assistant', cls='font-semibold text-sm'), Span(' Tricky...', cls='text-cyan-400 animate-pulse text-[15px] ml-1')]
    body = [Div(*parts, cls='flex items-center gap-1')]
    if text:
        body.append(Pre(text, cls='text-sm whitespace-pre-wrap mt-1 opacity-80'))
    return Div(*body, id=f'pending-{prompt_id}', cls='border-l-4 border-error pl-3 py-2 my-2 ml-8',
               hx_post=run_prompt_poll.to(id=prompt_id), hx_target=f'#pending-{prompt_id}',
               hx_swap='outerHTML', hx_trigger='load delay:300ms', **kw)

In [ ]:
#| export
def _finalize_prompt_reply(prompt_id:int, content:str, details:str|None) -> Cell|None:
    "Write a completed LLM reply into the Prompt's paired Assistant cell, creating it if it doesn't already exist. Shared tail end of the old (now removed) run_prompt_cell(): the context-building half lives in _start_prompt_run(), this is the cell-writing half."
    c = nb.get(prompt_id)
    if not c or c.ctype != 'prompt': return None
    i = nb.index(c.id)
    nxt = nb.cells[i+1] if i+1 < len(nb.cells) else None
    if nxt is not None and nxt.ctype == 'assistant':
        nxt.source, nxt.model, nxt.details = content, nb.active_model(), details
        nxt.ts = datetime.now().strftime('%I:%M:%S %p')
    else:
        nxt = nb.insert_at(i+1, 'assistant', content, model=nb.active_model(), details=details)
    return nxt

In [ ]:
#| export
@rt
def run_prompt_poll(id:int) -> FT|str:
    "Poll a streaming Prompt reply -- returns the current partial text while still running (re-triggering itself), or finalizes into the real Assistant cell once done."
    global _prompt_state
    st = _prompt_state
    if st is None or st.cell_id != id:
        return ''  # stale poll (e.g. a second tab, or the reply was already finalized) -- nothing to do
    if not st.done:
        return pending_prompt_cell(id, ''.join(st.buffer))
    content, details = st.blocks
    _prompt_state = None
    c2 = _finalize_prompt_reply(id, content, details)
    return render_cell(c2) if c2 else ''

In [ ]:
#| export
def add_tool(fn:callable) -> callable:
    "Register `fn` as a tool the LLM can call on future Prompt-cell runs. Also usable as a decorator: `@add_tool`."
    if not callable(fn):
        raise TypeError(f'add_tool() expects a callable, got {fn!r}')
    if fn not in nb.tools:
        nb.tools.append(fn)
    return fn

In [ ]:
#| export
_shell.push({'nb': nb, 'add_tool': add_tool})  # so code cells can call add_tool(...)/inspect nb directly, no import needed

def _push_tools() -> None:
    "Push every currently-selected tool function into the shared shell's namespace, keyed by name, so code the model suggests calling one of them (per tools_system_prompt(), see llms.py) is actually runnable as-is, no import needed. Called before each prompt run and after a kernel restart -- both leave the namespace out of sync with the current tool selection."
    tools = get_tool_list(nb.tool_selection) + nb.tools
    _shell.push({fn.__name__: fn for fn in tools})

_push_tools()  # populate the namespace once at startup, matching the current (default) tool selection

In [ ]:
#| export
def llm_context(nb:Notebook, cur_id:int|None=None) -> str:
    "Exactly what a real model would receive: the visible cells up through `cur_id` (default: all), in order."
    cutoff = len(nb.cells)
    if cur_id is not None:
        i = nb.index(cur_id)
        if i is not None: cutoff = i + 1
    return '\n'.join(f'[{c.ctype}] {c.source}' for c in nb.cells[:cutoff] if c.visible)


In [ ]:
#| export
def stub_reply(nb:Notebook, prompt:str) -> str:
    "Fake/placeholder LLM reply used when no model is available (lets you still test the GUI)."
    n = sum(c.visible for c in nb.cells)
    # TODO: use the prompt_llm routine instead to get real llm interaction
    return (f'(stub) I can see {n} visible cell(s). You said: '
            f'"{prompt.strip()}". Wire a real model into stub_reply() later.')


In [ ]:
#| export
def ensure_models() -> None:
    "Populate nb.models (info dicts, see get_model_list()) plus default picks for the standard and reasoning models, tolerating an unreachable local LLM server. The reasoning default is whichever model first advertises Ollama's 'thinking' capability, if any -- see brain_menu()."
    try:
        nb.models = get_model_list()
    except Exception:
        nb.models = []
    preferred = next((m for m in nb.models if _PREFERRED_MODEL_SUBSTR in m['id']), None)
    nb.standard_model = preferred['id'] if preferred else (nb.models[0]['id'] if nb.models else None)
    reasoning = next((m for m in nb.models if 'thinking' in m.get('capabilities', [])), None)
    nb.reasoning_model = reasoning['id'] if reasoning else None

In [ ]:
#| export
try:
    @app.on_event('startup')
    def _load_models() -> None:
        "Fetch the local model list once, when the real server actually starts up."
        ensure_models()
        start_plugins()  # spawn each registered plugin's background task (e.g. SystemMonitor's sampler) -- only on a real boot, never during import/tests
except:
    print("WARNING: Can't test this cell in notebook, no app")

## Rendering

Each type gets a colored left border (matching the SolveIt screenshot: raw=yellow,
code=blue, note=green, prompt/assistant=red). The selected cell gets a ring;
hidden-from-LLM cells are dimmed. Note cells render as markdown via the `.marked`
class (KaTeX, images, HTML).

In [ ]:
#| export
BORDER = {'raw':'border-warning', 'code':'border-info', 'note':'border-success',
          'prompt':'border-error', 'assistant':'border-error'}  # left-border color per cell type


In [ ]:
#| export
# heroicons (outline, 1.5 stroke) -- https://heroicons.com
# Each icon is a tuple of paths; a path can be a plain 'd' string, or (d, scale) to render that
# one sub-path in its own scaled group (about the 24x24 center) -- used for x-circle/play-circle
# so their inner glyph can be enlarged without also blowing up the surrounding circle.
ICONS = {
    'copy':          ('M8.25 9V5.25A2.25 2.25 0 0 1 10.5 3h6a2.25 2.25 0 0 1 2.25 2.25v13.5A2.25 2.25 0 0 1 16.5 21h-6a2.25 2.25 0 0 1-2.25-2.25V15m-3 0-3-3m0 0 3-3m-3 3H15',),
    'eye':           ('M2.036 12.322a1.012 1.012 0 0 1 0-.639C3.423 7.51 7.36 4.5 12 4.5c4.638 0 8.573 3.007 9.963 7.178.07.207.07.431 0 .639C20.577 16.49 16.64 19.5 12 19.5c-4.638 0-8.573-3.007-9.963-7.178Z',
                       'M15 12a3 3 0 1 1-6 0 3 3 0 0 1 6 0Z'),
    'eye-slash':     ('M3.98 8.223A10.477 10.477 0 0 0 1.934 12C3.226 16.338 7.244 19.5 12 19.5c.993 0 1.953-.138 2.863-.395M6.228 6.228A10.451 10.451 0 0 1 12 4.5c4.756 0 8.773 3.162 10.065 7.498a10.522 10.522 0 0 1-4.293 5.774M6.228 6.228 3 3m3.228 3.228 3.65 3.65m7.894 7.894L21 21m-3.228-3.228-3.65-3.65m0 0a3 3 0 1 0-4.243-4.243m4.242 4.242L9.88 9.88',),
    'trash':         ('m14.74 9-.346 9m-4.788 0L9.26 9m9.968-3.21c.342.052.682.107 1.022.166m-1.022-.165L18.16 19.673a2.25 2.25 0 0 1-2.244 2.077H8.084a2.25 2.25 0 0 1-2.244-2.077L4.772 5.79m14.456 0a48.108 48.108 0 0 0-3.478-.397m-12 .562c.34-.059.68-.114 1.022-.165m0 0a48.11 48.11 0 0 1 3.478-.397m7.5 0v-.916c0-1.18-.91-2.164-2.09-2.201a51.964 51.964 0 0 0-3.32 0c-1.18.037-2.09 1.022-2.09 2.201v.916m7.5 0a48.667 48.667 0 0 0-7.5 0',),
    'arrow-up':      ('M4.5 10.5 12 3m0 0 7.5 7.5M12 3v18',),
    'arrow-down':    ('M19.5 13.5 12 21m0 0-7.5-7.5M12 21V3',),
    'arrow-path':    ('M16.023 9.348h4.992v-.001M2.985 19.644v-4.992m0 0h4.992m-4.993 0 3.181 3.183a8.25 8.25 0 0 0 13.803-3.7M4.031 9.865a8.25 8.25 0 0 1 13.803-3.7l3.181 3.182m0-4.991v4.99',),
    'x-circle':      (('m9.75 9.75 4.5 4.5m0-4.5-4.5 4.5', 1.35),
                       'M21 12a9 9 0 1 1-18 0 9 9 0 0 1 18 0Z'),
    'play':          ('M5.25 5.653c0-.856.917-1.398 1.667-.986l11.54 6.347a1.125 1.125 0 0 1 0 1.972l-11.54 6.347a1.125 1.125 0 0 1-1.667-.986V5.653Z',),
    'play-circle':   ('M21 12a9 9 0 1 1-18 0 9 9 0 0 1 18 0Z',
                       ('M15.91 11.672a.375.375 0 0 1 0 .656l-5.603 3.113a.375.375 0 0 1-.557-.328V8.887c0-.286.307-.466.557-.327l5.603 3.112Z', 1.35)),
    'bookmark':      ('M17.593 3.322c1.1.128 1.907 1.077 1.907 2.185V21L12 17.25 4.5 21V5.507c0-1.108.806-2.057 1.907-2.185a48.507 48.507 0 0 1 11.186 0Z',),
    'bars-3':        ('M3.75 6.75h16.5M3.75 12h16.5m-16.5 5.25h16.5',),
    'folder':        ('M2.25 12.75V12A2.25 2.25 0 0 1 4.5 9.75h15A2.25 2.25 0 0 1 21.75 12v.75m-8.69-6.44-2.12-2.12a1.5 1.5 0 0 0-1.061-.44H4.5A2.25 2.25 0 0 0 2.25 6v12a2.25 2.25 0 0 0 2.25 2.25h15A2.25 2.25 0 0 0 21.75 18V9a2.25 2.25 0 0 0-2.25-2.25h-5.379a1.5 1.5 0 0 1-1.06-.44Z',),
    'document-text': ('M19.5 14.25v-2.625a3.375 3.375 0 0 0-3.375-3.375h-1.5A1.125 1.125 0 0 1 13.5 7.125v-1.5a3.375 3.375 0 0 0-3.375-3.375H8.25m0 12.75h7.5m-7.5 3H12M10.5 2.25H5.625c-.621 0-1.125.504-1.125 1.125v17.25c0 .621.504 1.125 1.125 1.125h12.75c.621 0 1.125-.504 1.125-1.125V11.25a9 9 0 0 0-9-9Z',),
    'question-mark-circle': ('M9.879 7.519c1.171-1.025 3.071-1.025 4.242 0 1.172 1.025 1.172 2.687 0 3.712-.203.179-.43.326-.67.442-.745.361-1.45.999-1.45 1.827v.75M21 12a9 9 0 1 1-18 0 9 9 0 0 1 18 0Zm-9 5.25h.008v.008H12v-.008Z',),
    'wrench-screwdriver': ('M11.42 15.17 17.25 21A2.652 2.652 0 0 0 21 17.25l-5.877-5.877M11.42 15.17l2.496-3.03c.317-.384.74-.626 1.208-.766M11.42 15.17l-4.655 5.653a2.548 2.548 0 1 1-3.586-3.586l6.837-5.63m5.108-.233c.55-.164 1.163-.188 1.743-.14a4.5 4.5 0 0 0 4.486-6.336l-3.276 3.277a3.004 3.004 0 0 1-2.25-2.25l3.276-3.276a4.5 4.5 0 0 0-6.336 4.486c.091 1.076-.071 2.264-.904 2.95l-.102.085m-1.745 1.437L5.909 7.5H4.5L2.25 3.75l1.5-1.5L7.5 4.5v1.409l4.26 4.26m-1.745 1.437 1.745-1.437m6.615 8.206L15.75 15.75M4.867 19.125h.008v.008h-.008v-.008Z',),
    # lucide-static (ISC license) brain.svg -- heroicons has no literal brain icon; kept in the same plain-'d'-tuple, stroke='currentColor' style as the rest of ICONS. Every path gets the same 0.85 scale (see Icon()'s (d,scale) handling) because lucide's glyph fills ~83% of its 24x24 box vs ~75-81% for our heroicons (all centered at (12,12) like this one is) -- left at 1.0 it visually reads heavier/bigger than its neighbors despite an identical button box.
    'brain': (('M12 18V5', .85), ('M15 13a4.17 4.17 0 0 1-3-4 4.17 4.17 0 0 1-3 4', .85), ('M17.598 6.5A3 3 0 1 0 12 5a3 3 0 1 0-5.598 1.5', .85),
              ('M17.997 5.125a4 4 0 0 1 2.526 5.77', .85), ('M18 18a4 4 0 0 0 2-7.464', .85), ('M19.967 17.483A4 4 0 1 1 12 18a4 4 0 1 1-7.967-.517', .85),
              ('M6 18a4 4 0 0 1-2-7.464', .85), ('M6.003 5.125a4 4 0 0 0-2.526 5.77', .85)),
}  # heroicon name -> tuple of SVG path 'd' attributes


In [ ]:
#| export
def _svg_icon(paths, cls:str='size-4', filled:bool=False) -> FT:
    "Render a tuple of ICONS-style path data (plain 'd' strings and/or (d, scale) pairs) as an inlined SVG with stroke='currentColor' -- outline style (fill='none') by default, or a solid fill='currentColor' silhouette if `filled` -- so both stroke and any fill track the button's text color via `cls`. An (d, scale) pair renders that sub-path in its own group, scaled about the 24x24 viewBox center, so it can be enlarged independently of the rest (e.g. x-circle's X). Shared by Icon() (built-in ICONS) and plugin icons (see plugins.py / _plugin_button), so a plugin's SVG renders identically to a built-in one."
    def render(item):
        d, scale = item if isinstance(item, tuple) else (item, 1)
        p = SvgPath(stroke_linecap='round', stroke_linejoin='round', d=d)
        return SvgG(p, transform=f'translate(12,12) scale({scale}) translate(-12,-12)') if scale != 1 else p
    return fh.Svg(*[render(item) for item in paths],
                  xmlns='http://www.w3.org/2000/svg', fill='currentColor' if filled else 'none', viewbox='0 0 24 24',
                  stroke_width='1.5', stroke='currentColor', cls=cls)

In [ ]:
#| export
def Icon(name:str, cls:str='size-4', filled:bool=False) -> FT:
    "A heroicons SVG by name (see ICONS), inlined so `stroke='currentColor'` (and, if `filled`, `fill='currentColor'` too) matches the button's text color. Thin wrapper over _svg_icon()."
    return _svg_icon(ICONS[name], cls, filled)

In [ ]:
#| export
def IconBtn(name:str, title:str, **kw) -> FT:
    "A small ghost-style button showing heroicon `name`, with a hover tooltip of `title`."
    return fh.Button(Icon(name), cls='btn btn-sm btn-ghost tooltip tooltip-bottom', data_tip=title, **kw)


In [ ]:
#| export
def cell_toolbar(c:Cell) -> FT:
    "The row of icon buttons (copy, export toggle, visibility, run, move, delete) shown in a cell's header."
    copy_btn = fh.Button(Icon('copy'), id=f'copy-{c.id}', data_tip='Copy to clipboard', type='button',
                         cls='btn btn-sm btn-ghost tooltip tooltip-bottom', data_src=c.source,
                         onclick=f"boopCopy({c.id}, '{c.ctype}')")
    btns = [copy_btn]
    if c.ctype == 'code':
        btns.append(fh.Button(Icon('bookmark', cls='size-4 icon-fillable' + (' icon-filled' if c.export else '')),
                              id=f'export-btn-{c.id}', data_tip='Exported (#| export)' if c.export else 'Not exported',
                              type='button', onclick=f'boopToggleExport({c.id})',
                              # green (text-success, same as note cells' border-success), not red -- red reads as
                              # "something's wrong" when export=True is actually the intended, working state. A solid
                              # fill (not just the outline) on top of that, since a green outline alone read as less
                              # noticeable than the red outline it replaced -- solid fill draws the eye more reliably.
                              # onclick (not hx_post/hx_target='outerHTML') -- boopToggleExport() flips the CSS class
                              # instantly and persists in the background (see edit.js), instead of swapping the whole
                              # cell just to change one icon's fill/color, which was visibly flickering the page.
                              cls='btn btn-sm btn-ghost tooltip tooltip-bottom' + (' text-success' if c.export else '')))
    # Both eye/eye-slash icons render always, one 'hidden' -- a CSS class can't morph one icon's
    # path data into another's the way it can flip a fill color, so we ship both and toggle which
    # is shown. Same instant-CSS/no-outerHTML-swap idea as the export bookmark above -- see
    # boopToggleVis() in edit.js, which also toggles #cell-{id}'s own dimming (opacity-70) directly.
    btns.append(fh.Button(
        Icon('eye', cls='size-4 vis-eye' + ('' if c.visible else ' hidden')),
        Icon('eye-slash', cls='size-4 vis-eye-slash' + (' hidden' if c.visible else '')),
        id=f'vis-btn-{c.id}', data_tip='Hide from LLM' if c.visible else 'Show to LLM',
        type='button', onclick=f'boopToggleVis({c.id})',
        # red (text-error) while hidden -- unlike the export bookmark, red is the right call here:
        # a hidden cell IS being excluded/withheld from the LLM, which is exactly what red usually
        # signals, not a false alarm the way it was for the (working-as-intended) export flag.
        cls='btn btn-sm btn-ghost tooltip tooltip-bottom' + ('' if c.visible else ' text-error')))
    if c.ctype == 'code':
        # hx_post (not onclick=boopSave) so Run works whether or not the cell is currently being
        # edited -- boopSave() requires a live CodeMirror/textarea, which a static (non-selected)
        # code cell doesn't have.
        btns.append(IconBtn('play', 'Run', hx_post=run_cell.to(id=c.id), hx_target=f'#cell-{c.id}', hx_swap='outerHTML'))
    elif c.ctype == 'prompt':
        # Targeted swap (not the whole #notebook) so re-running a prompt cell mid-notebook doesn't
        # blow away scroll position: swap the existing Assistant reply if there is one, else insert
        # a pending placeholder right after this cell.
        i = nb.index(c.id)
        nxt = nb.cells[i+1] if i+1 < len(nb.cells) else None
        if nxt is not None and nxt.ctype == 'assistant':
            p_target, p_swap = f'#cell-{nxt.id}', 'outerHTML'
        else:
            p_target, p_swap = f'#cell-{c.id}', 'afterend'
        btns.append(IconBtn('play', 'Run', hx_post=run_cell.to(id=c.id), hx_target=p_target, hx_swap=p_swap))
    # Move/delete mutate the DOM out-of-band (see move_cell/del_cell), so these buttons don't
    # need a real hx-target -- swap='none' means htmx applies the response's OOB directives only.
    btns += [
        IconBtn('arrow-up', 'Move up',   hx_post=move_cell.to(id=c.id, delta=-1), hx_swap='none'),
        IconBtn('arrow-down', 'Move down', hx_post=move_cell.to(id=c.id, delta=1),  hx_swap='none'),
        IconBtn('trash', 'Delete', hx_post=del_cell.to(id=c.id), hx_swap='none'),
    ]
    return Div(*btns, cls='flex gap-1 ml-auto')


In [ ]:
#| export
def type_dropdown(c:Cell) -> FT:
    "Click the cell-type word to switch it (code/note/prompt/raw). Scoped to just this cell."
    if c.ctype == 'assistant':
        return Span('Assistant', cls='font-semibold text-sm')
    opts = [Li(fh.A(t.capitalize(), hx_post=set_ctype.to(id=c.id, t=t),
                    hx_target=f'#cell-{c.id}', hx_swap='outerHTML'))
            for t in CTYPES if t != c.ctype]
    return Div(
        Div(c.ctype.capitalize(), tabindex='0', role='button',
            cls='font-semibold text-sm cursor-pointer'),
        Ul(*opts, tabindex='0', cls='dropdown-content menu bg-base-200 rounded-box z-10 w-28 p-1 shadow'),
        cls='dropdown dropdown-bottom')


In [ ]:
#| export
def cell_header(c:Cell, running:bool=False) -> FT:
    "The top row of a cell: type dropdown, id/timestamp, and the toolbar. `running=True` (used by pending_code_cell() while a background execution is still in flight) adds a pulsing 'Running...' indicator that disappears once the cell finishes, whether it succeeded or errored. flex-wrap lets the toolbar drop to its own line on narrow (mobile) viewports instead of squeezing/overlapping."
    rest = f': {c.id}' + (f' ({c.ts})' if c.ctype in ('code','assistant') else '')
    # Model name lives in the Reply details block now (see Cell.details) -- only fall back to
    # showing it here for older cells saved before that existed, so the header stays clean.
    if c.ctype == 'assistant' and c.model and not c.details: rest += f' \xb7 {c.model}'
    parts = [rest]
    if running:
        parts.append(Span(' Running...', cls='text-cyan-400 animate-pulse font-bold'))
    return Div(type_dropdown(c),
               Span(*parts, cls='font-semibold text-sm cursor-pointer flex-1',
                    hx_post=select.to(id=c.id), hx_target=f'#cell-{c.id}', hx_swap='outerHTML'),
               cell_toolbar(c), cls='flex flex-wrap items-center gap-2 mb-1')


In [ ]:
#| export
def cell_body(c:Cell) -> FT:
    "Note/prompt/assistant render as markdown; raw is bare text. Code cells never reach here -- render_cell() routes them to code_view()/code_editor(). An Assistant cell with call metadata (Cell.details) shows it in a collapsed <details> block above the reply."
    details = NotStr(c.details) if (c.ctype == 'assistant' and c.details) else None
    if not c.source.strip():  # otherwise an empty cell has nothing visible to click to start editing
        main = Span('(empty -- click to edit)', cls='opacity-40 italic text-sm')
    elif c.ctype in ('note', 'prompt', 'assistant'):  # markdown + KaTeX + HTML + images + highlighted code fences
        main = Div(c.source, cls='marked prose max-w-none')
    else:
        main = Pre(c.source, cls='font-mono text-sm whitespace-pre-wrap')
    return Div(details, main, cls='flex flex-col gap-1') if details is not None else main


In [ ]:
#| export
def _cell_outer(c:Cell, *content, oob=None, **extra) -> FT:
    "The bordered wrapper div common to every cell, regardless of type or edit state. `oob`, if given, marks this div for an htmx out-of-band swap: True for a plain id-matched replace, or an 'hx-swap-oob' spec string (e.g. 'afterend:#cell-5') for a positional insert/replace elsewhere in the DOM. `extra` kwargs (e.g. hx_post/hx_trigger) are forwarded straight to the Div -- see pending_code_cell()."
    dim    = '' if c.visible else 'opacity-70'  # split the difference between fully dim and fully visible -- 40 made it too hard to read/edit a hidden cell's content
    indent = 'ml-8' if c.ctype == 'assistant' else ''
    ring   = 'ring-2 ring-primary ring-offset-2 ring-offset-base-100 rounded' if c.id == nb.selected else ''
    kw = {'hx_swap_oob': 'true' if oob is True else oob} if oob else {}
    return Div(*content, id=f'cell-{c.id}',
               cls=f'border-l-4 {BORDER[c.ctype]} pl-3 py-2 my-2 {dim} {indent} {ring}', **kw, **extra)


In [ ]:
#| export
def render_output_blocks(blocks:list[dict]) -> list[FT]:
    "Render each of a code cell's output blocks (see Cell.output / run_code()) to its appropriate FT: an <img> for images, raw markup for HTML/SVG, client-side-rendered markdown for text/markdown (same '.marked' pipeline as note cells), pretty-printed for JSON, and a plain (ANSI-aware) <pre> for stream/error text."
    out = []
    for b in blocks:
        mime, data = b.get('mime'), b['data']
        if mime == 'image/png':
            out.append(Img(src=f'data:image/png;base64,{data}', cls='max-w-full mt-1'))
        elif mime == 'image/svg+xml':
            out.append(Div(NotStr(data), cls='mt-1'))
        elif mime == 'text/html':
            out.append(Div(NotStr(data), cls='mt-1'))
        elif mime == 'text/markdown':
            out.append(Div(data, cls='marked prose max-w-none mt-1'))
        elif mime == 'application/json':
            out.append(Pre(json.dumps(json.loads(data), indent=2), cls='text-sm mt-1 overflow-x-auto'))
        else:
            cls = 'ansi-out text-sm mt-1 whitespace-pre overflow-x-auto' + (' text-error' if b['type'] == 'error' else '')
            out.append(Pre(data, cls=cls))
    return out

In [ ]:
#| export
def code_view(c:Cell) -> FT:
    "Static, syntax-highlighted (no live CodeMirror) view of a code cell -- click to load the real editor. Keeping non-focused cells static is what makes theme switches etc. fast on notebooks with many code cells."
    if not c.source.strip():
        parts = [Span('(empty -- click to edit)', cls='opacity-40 italic text-sm')]
    else:
        parts = [Pre(Code(c.source, cls='language-python'), cls='text-sm overflow-x-auto')]
    if c.output:
        parts += render_output_blocks(c.output)
    return Div(*parts, cls='cursor-text', hx_get=edit_cell.to(id=c.id), hx_target=f'#cell-{c.id}', hx_swap='outerHTML')


In [ ]:
#| export
def code_editor(c:Cell) -> FT:
    "The live CodeMirror editor for a code cell -- only rendered for the cell currently being edited. Shift/Ctrl/Cmd+Enter or the play button runs. c.source never contains the '#| export' pragma -- see Cell.export / the bookmark toggle in cell_toolbar."
    ta = Textarea(c.source, name='source', id=f'ta-{c.id}',
                  rows=str(max(2, c.source.count(chr(10)) + 1)),
                  cls='textarea textarea-bordered w-full font-mono',
                  data_cm='code', data_cid=str(c.id))
    parts = [Form(ta, hx_post=save_cell.to(id=c.id), hx_target=f'#cell-{c.id}', hx_swap='outerHTML')]
    if c.output:
        parts += render_output_blocks(c.output)
    return Div(*parts)


In [ ]:
#| export
def render_cell(c:Cell, oob=None) -> FT:
    "Every cell type renders statically and opens its editor on click; only the actively-edited cell gets a live widget (CodeMirror for code, a plain textarea otherwise). `oob` is forwarded to `_cell_outer` -- see there."
    if c.ctype == 'code':
        return _cell_outer(c, cell_header(c), code_editor(c) if c.id == nb.selected else code_view(c), oob=oob)
    body = cell_body(c)
    body = Div(body, cls='cursor-text',
                   hx_get=edit_cell.to(id=c.id), hx_target=f'#cell-{c.id}', hx_swap='outerHTML')
    return _cell_outer(c, cell_header(c), body, oob=oob)


In [ ]:
#| export
def _oob(spec:str, *content) -> FT:
    "Wrap `content` in a throwaway div carrying a positional htmx out-of-band directive (e.g. 'afterend:#cell-5' or 'beforeend:#notebook'). Needed because htmx's positional OOB swaps (beforebegin/afterend/beforeend) insert only the *children* of the OOB-tagged element, not the element itself -- so anything that needs to land in the DOM with its own id intact (a cell div, a pending placeholder) must be nested one level below the OOB wrapper, not carry the hx-swap-oob attribute itself."
    return Div(*content, hx_swap_oob=spec)

In [ ]:
#| export
def render_cell_edit(c:Cell) -> FT:
    "Inline editor. Code cells use CodeMirror (Python highlight, no wrap); notes/raw use a textarea. Shift/Ctrl/Cmd+Enter saves."
    if c.ctype == 'code':
        return _cell_outer(c, cell_header(c), code_editor(c))
    ta = Textarea(c.source, name='source', id=f'ta-{c.id}',
                  rows=str(max(3, c.source.count(chr(10)) + 2)),
                  cls='textarea textarea-bordered w-full font-mono',
                  data_cm='edit', data_cid=str(c.id))
    buttons = Div(Button('Save', type='button', cls='btn btn-primary btn-xs',
                         onclick=f'boopSave({c.id})'),
                  Button('Cancel', type='button', cls='btn btn-ghost btn-xs',
                         hx_get=view_cell.to(id=c.id), hx_target=f'#cell-{c.id}', hx_swap='outerHTML'),
                  cls='flex gap-2 justify-end mt-1')
    form = Form(ta, buttons, hx_post=save_cell.to(id=c.id),
                hx_target=f'#cell-{c.id}', hx_swap='outerHTML')
    return _cell_outer(c, cell_header(c), form)

In [ ]:
#| export
def render_nb() -> FT:
    "Render every cell in the notebook, in order, inside the #notebook container div."
    return Div(*[render_cell(c) for c in nb.cells], id='notebook', cls='flex flex-col')

## Composer

The bottom bar: type tabs, a textarea, Submit. Picking a tab sets the type
server-side; Submit creates the cell (running it, if code; spawning an Assistant
reply, if prompt).

In [ ]:
#| export
def composer(draft:str='', oob:bool=False) -> FT:
    "The bottom-of-page input bar: type tabs, a source textarea, and a Boop (submit) button."
    tabs = [fh.A(t.capitalize(),
                 cls=f'tab {"tab-active" if nb.compose_type==t else ""}',
                 hx_post=set_type.to(t=t), hx_target='#composer', hx_swap='outerHTML')
            for t in CTYPES]
    ta_kw = {'data_cm':'composer'} if nb.compose_type=='code' else {}
    div_kw = {'hx_swap_oob':'true'} if oob else {}
    return Div(
        Div(*tabs, cls='tabs tabs-boxed'),
        Form(Textarea(draft, placeholder=f'{nb.compose_type} cell…', name='source',
                      id='compose-input', rows='3',
                      onkeydown="if((event.shiftKey||event.ctrlKey||event.metaKey)&&event.key==='Enter')"
                               "{event.preventDefault();this.form.requestSubmit();}",
                      cls='textarea textarea-bordered w-full font-mono', **ta_kw),
             Div(Button('Boop', type='button', onclick='boopComposerSubmit()', cls='btn btn-primary'), cls='flex justify-end mt-2'),
             hx_post=submit_cell, hx_target='#notebook', hx_swap='beforeend'),
        id='composer', cls='border-t border-base-300 pt-3 mt-4', **div_kw)


In [ ]:
#| export
def render_app(draft:str='') -> FT:
    "The whole notebook view: all cells plus the composer, wrapped in one container div."
    return Div(render_nb(), composer(draft), id='app')

## Routes

Composer/toolbar routes plus the command-mode routes driven by hotkeys
(`select`, `select_delta`, `insert`, `del_selected`, `settype_selected`).

In [ ]:
#| export
# ---- top menu / control bar ----
def theme_swap() -> FT:
    "DaisyUI sun/moon swap; drives boopApplyTheme (default dark)."
    # tooltip-right-align: this is the rightmost element in the navbar, flush against the viewport
    # edge, so a centered tooltip-bottom would overflow off-screen -- anchor its right edge instead.
    return NotStr('<label class="swap swap-rotate btn btn-ghost btn-circle btn-sm tooltip tooltip-bottom tooltip-right-align" data-tip="Toggle light/dark">'
      '<input type="checkbox" id="theme-toggle" onchange="boopThemeToggle(this)" checked />'
      '<svg class="swap-off h-5 w-5 fill-current" xmlns="http://www.w3.org/2000/svg" viewBox="0 0 24 24">'
      '<path d="M5.64,17l-.71.71a1,1,0,0,0,0,1.41,1,1,0,0,0,1.41,0l.71-.71A1,1,0,0,0,5.64,17ZM5,12a1,1,0,0,0-1-1H3a1,1,0,0,0,0,2H4A1,1,0,0,0,5,12Zm7-7a1,1,0,0,0,1-1V3a1,1,0,0,0-2,0V4A1,1,0,0,0,12,5ZM5.64,7.05a1,1,0,0,0,.7.29,1,1,0,0,0,.71-.29,1,1,0,0,0,0-1.41l-.71-.71A1,1,0,0,0,4.93,6.34Zm12,.29a1,1,0,0,0,.7-.29l.71-.71a1,1,0,1,0-1.41-1.41L17,5.64a1,1,0,0,0,0,1.41A1,1,0,0,0,17.66,7.34ZM21,11H20a1,1,0,0,0,0,2h1a1,1,0,0,0,0-2Zm-9,8a1,1,0,0,0-1,1v1a1,1,0,0,0,2,0V20A1,1,0,0,0,12,19ZM18.36,17A1,1,0,0,0,17,18.36l.71.71a1,1,0,0,0,1.41,0,1,1,0,0,0,0-1.41ZM12,6.5A5.5,5.5,0,1,0,17.5,12,5.51,5.51,0,0,0,12,6.5Zm0,9A3.5,3.5,0,1,1,15.5,12,3.5,3.5,0,0,1,12,15.5Z"/></svg>'
      '<svg class="swap-on h-5 w-5 fill-current" xmlns="http://www.w3.org/2000/svg" viewBox="0 0 24 24">'
      '<path d="M21.64,13a1,1,0,0,0-1.05-.14,8.05,8.05,0,0,1-3.37.73A8.15,8.15,0,0,1,9.08,5.49a8.59,8.59,0,0,1,.25-2A1,1,0,0,0,8,2.36,10.14,10.14,0,1,0,22,14.05,1,1,0,0,0,21.64,13Zm-9.5,6.69A8.14,8.14,0,0,1,7.08,5.22v.27A10.15,10.15,0,0,0,17.22,15.63a9.79,9.79,0,0,0,2.1-.22A8.11,8.11,0,0,1,12.14,19.73Z"/></svg></label>')


In [ ]:
#| export
def fname_display() -> FT:
    "The clickable filename shown in the top bar; click to rename."
    return Span(nb.name, id='fname', data_tip='Click to rename',
                cls='cursor-pointer font-mono opacity-80 hover:opacity-100 tooltip tooltip-bottom',
                hx_get=rename_form, hx_target='#fname', hx_swap='outerHTML')


In [ ]:
#| export
@rt
def rename_form() -> FT:
    "Swap the filename display for a text input, focused and pre-selected, to rename the notebook."
    return Form(Input(value=nb.name, name='name',
                      cls='input input-sm input-bordered font-mono',
                      onkeydown="if(event.key===\'Escape\'){this.form.requestSubmit();}"),
                Script("var i=document.querySelector(\'#fname input\'); if(i){i.focus();i.select();}"),
                id='fname', hx_post=rename, hx_target='#fname', hx_swap='outerHTML')

In [ ]:
#| export
_CAPABILITY_EMOJI = {'vision': '👁️', 'tools': '🔧', 'thinking': '🧠'}  # capability (from get_ollama_list()'s 'capabilities') -> emoji shown after a model's name in dropdowns; capabilities we don't care about (completion -- basically universal, insert, embedding, ...) are silently ignored, and a model with none of these three gets no emoji at all

In [ ]:
#| export
def _model_label(m:dict) -> str:
    "A model's dropdown label: its bare name, plus a trailing run of capability emojis (see _CAPABILITY_EMOJI) if it has any worth flagging."
    emoji = ''.join(_CAPABILITY_EMOJI[c] for c in m.get('capabilities', []) if c in _CAPABILITY_EMOJI)
    return f"{m['model']}  {emoji}" if emoji else m['model']

In [ ]:
#| export
def _model_select(models:list[dict], active_id:str|None, route) -> FT:
    "An inline, DOM-contained list of `models` as clickable rows -- NOT a native <select>, whose option list the browser renders as OS chrome *outside* the page DOM, which breaks brain_menu()'s hover/focus-within visibility the moment the pointer moves onto it (the whole menu vanishes before you can pick anything). Clicking a row posts the chosen model id to `route`, which re-renders the whole brain menu (hx_target=#brain-menu) so the new pick's checkmark shows. Shared by the Standard/Reasoning pickers."
    rows = []
    for m in models:
        active = (m['id'] == active_id)
        rows.append(fh.Li(fh.A(
            Span('✓' if active else '', cls='inline-block w-3 text-cyan-400'),
            Span(_model_label(m)),
            hx_post=route.to(model=m['id']), hx_target='#brain-menu', hx_swap='outerHTML',
            cls='flex items-center gap-1 py-0.5' + (' font-semibold' if active else ''))))
    return fh.Ul(*rows, cls='menu menu-sm p-0 gap-0 w-full max-h-64 overflow-y-auto flex-nowrap')

In [ ]:
#| export
def _dropdown_width_px(models:list[dict]) -> int:
    "Estimate a comfortable width (px) for brain_menu()'s popup from the longest model label (name + capability emojis) among `models` -- a fixed width clipped the select boxes (arrow overlapping the text) whenever a name+emoji combo ran longer than expected, applied via inline style (see brain_menu()) since a computed w-[Npx] class never works against this app's precompiled static tailwind.css. ~9px/char is a rough per-character width at this font size; the +100 covers the select's own padding/border/dropdown-arrow chrome plus the popup's own outer p-2 padding. Clamped to a sane range either way."
    longest = max((len(_model_label(m)) for m in models), default=20)
    return max(220, min(380, 45 + longest * 8))

In [ ]:
#| export
_EFFORT_LEVELS = ('l', 'm', 'h')

def brain_menu(icon_cls:str) -> FT:
    "Hover menu (styled like tools_menu) for picking the Standard and Reasoning models, SolveIt-style -- the Reasoning picker only lists models advertising Ollama's 'thinking' capability (see get_ollama_list()/ensure_models()), with an L/M/H effort control alongside it (passed through as Chat(...)(think=...) -- see stream_llm_reply()). The brain icon itself doubles as a toggle: click it to switch whether the reasoning or standard model actually answers Prompt cells (nb.use_reasoning/active_model()) -- its SVG strokes turn cyan (matching the 'Tricky...' streaming indicator's color, our existing 'this is thinking' cue) while on. Re-renders itself wholesale on toggle (hx_target=self) since the button's own color has to change along with the menu."
    reasoning_models = [m for m in nb.models if 'thinking' in m.get('capabilities', [])]
    # DaisyUI's .menu auto-flexes a li's direct children into a row -- explicit flex-col here
    # overrides that so Standard/Reasoning/Effort stack vertically instead of piling up sideways.
    rows = [Div('Model Selection', cls='font-bold text-sm mb-1'),
            Div('Standard model', cls='text-sm'),
            _model_select(nb.models, nb.standard_model, set_standard_model) if nb.models else Span('no models', cls='text-xs opacity-50'),
            Div('Reasoning model', cls='text-sm mt-2 text-cyan-400'),  # cyan reinforces the same 'this is thinking' cue as the brain icon's own toggled color and the 'Tricky...' streaming indicator
            _model_select(reasoning_models, nb.reasoning_model, set_reasoning_model) if reasoning_models else Span('none available', cls='text-xs opacity-50')]
    if reasoning_models:
        rows.append(Div(
            Span('Effort', cls='text-xs font-semibold mr-2'),
            *[Label(Input(type='radio', name='effort', checked=(nb.reasoning_effort == lvl), cls='radio radio-xs',
                          hx_post=set_reasoning_effort.to(effort=lvl), hx_swap='none'),
                    Span(lvl.upper(), cls='text-xs ml-1'), cls='flex items-center gap-1 mr-3 cursor-pointer')
              for lvl in _EFFORT_LEVELS],
            cls='flex items-center mt-2'))
    brain_cls = 'btn btn-ghost btn-circle btn-sm' + (' text-cyan-400' if nb.use_reasoning else '')
    width_px = _dropdown_width_px(nb.models)
    return Div(
        fh.Button(Icon('brain', cls=icon_cls), cls=brain_cls,
                  hx_post=toggle_reasoning, hx_target='#brain-menu', hx_swap='outerHTML'),
        # width is estimated from the longest model label (see _dropdown_width_px) rather than
        # fixed, so a long name+emoji combo doesn't overflow its select box -- set via inline style,
        # NOT a w-[Npx] utility class: this app serves a precompiled static tailwind.css (see
        # _tw_header()) built by scanning literal class strings in the source, so a class built from
        # an f-string with a runtime-varying number never has a matching CSS rule and silently does
        # nothing. `left:50%; transform:translateX(-50%)` (centering -- DaisyUI's dropdown only
        # offers left/right anchoring, neither of which centers on the trigger) is inline for the
        # same reason.
        Ul(Li(Div(*rows, cls='flex flex-col gap-2 p-2')), tabindex='0',
           style=f'left:50%; transform:translateX(-50%); width:{width_px}px;',
           cls='dropdown-content menu bg-base-200 rounded-box z-10 p-1 shadow'),
        id='brain-menu', cls='dropdown dropdown-hover dropdown-bottom')


In [ ]:
#| export
def _apply_active_model(prev_active:str|None) -> None:
    "After changing standard_model/reasoning_model/use_reasoning, stop whichever Ollama model was previously active (see nb.active_model()) if the switch actually changed it -- frees its VRAM, same as the old single-model set_model() used to do."
    new_active = nb.active_model()
    if prev_active and prev_active != new_active and prev_active.startswith('ollama/'):
        try: subprocess.run(['ollama', 'stop', prev_active.removeprefix('ollama/')], capture_output=True, timeout=5)
        except Exception: pass

In [ ]:
#| export
@rt
def set_standard_model(model:str) -> FT:
    "Change the Standard-model pick in the brain menu, then re-render the menu so the new pick's checkmark shows (see _model_select). Only affects Prompt answers directly if the reasoning toggle is currently off -- see nb.active_model()."
    if any(m['id'] == model for m in nb.models) and model != nb.standard_model:
        prev_active = nb.active_model()
        nb.standard_model = model
        _apply_active_model(prev_active)
    return brain_menu(_TOPBAR_ICON_CLS_LG)

In [ ]:
#| export
@rt
def set_reasoning_model(model:str) -> FT:
    "Change the Reasoning-model pick in the brain menu (restricted there to 'thinking'-capable models), then re-render the menu so the new pick's checkmark shows (see _model_select). Only affects Prompt answers directly if the reasoning toggle is currently on -- see nb.active_model()."
    if any(m['id'] == model for m in nb.models) and model != nb.reasoning_model:
        prev_active = nb.active_model()
        nb.reasoning_model = model
        _apply_active_model(prev_active)
    return brain_menu(_TOPBAR_ICON_CLS_LG)

In [ ]:
#| export
@rt
def toggle_reasoning() -> FT:
    "The brain-icon click: flip whether the reasoning or standard model answers Prompt cells. Returns the whole re-rendered brain_menu(), since the button itself needs to pick up its new highlighted state."
    prev_active = nb.active_model()
    nb.use_reasoning = not nb.use_reasoning
    _apply_active_model(prev_active)
    return brain_menu(_TOPBAR_ICON_CLS_LG)

In [ ]:
#| export
@rt
def set_reasoning_effort(effort:str) -> str:
    "Set the L/M/H reasoning-effort radio in the brain menu -- only meaningful while the reasoning model is in use (see nb.reasoning_effort/stream_llm_reply)."
    nb.reasoning_effort = effort
    return ''

In [ ]:
#| export
def _file_menu_items() -> list:
    "The New/Open/Save/Download/Restart Server actions shared by file_menu() (the hamburger dropdown) and context_menu() (the right-click popup) -- one source of truth so the two stay identical."
    return [
        Li(fh.A('New', href=new_notebook.to(),
                onclick="return confirm('Discard the current notebook and start a new one?')")),
        Li(fh.A('Open', onclick="document.getElementById('file-modal').showModal()",
                hx_get=browse.to(path=None if nb.name == 'untitled' else str(Path(nb.name).parent)),
                hx_target='#file-browser-body', hx_swap='innerHTML')),
        Li(fh.A('Save', href='javascript:void(0)', onclick='boopSaveNotebook()')),
        Li(fh.A('Download', href=download.to())),
        Li(fh.A('Restart Server', href='javascript:void(0)', onclick='boopRestartServer()')),
        Li(fh.A('Shutdown', href='javascript:void(0)', onclick='boopShutdownServer()')),
    ]

In [ ]:
#| export
def file_menu() -> FT:
    "Hamburger dropdown: New / Open (file browser) / Save / Download / Restart Server."
    return Div(
        Div(Icon('bars-3'), tabindex='0', role='button', cls='btn btn-ghost btn-circle btn-sm'),
        Ul(*_file_menu_items(), tabindex='0', cls='dropdown-content menu bg-base-200 rounded-box z-10 w-40 p-1 shadow'),
        cls='dropdown dropdown-bottom')

In [ ]:
#| export
def context_menu() -> FT:
    "The same New/Open/Save/Download/Restart Server menu as file_menu(), but shown wherever you right-click anywhere on the page (see boopContextMenu in theme.js), for people who reach for a right-click before the hamburger or the 's' hotkey."
    return Ul(*_file_menu_items(), id='context-menu', tabindex='0',
              cls='menu bg-base-200 rounded-box z-50 w-40 p-1 shadow fixed hidden')


In [ ]:
#| export
def file_browser_modal() -> FT:
    "The (initially empty/hidden) dialog that hosts the file-browser listing, opened by file_menu()'s Open item."
    return Dialog(
        Div(
            Div('Open notebook', cls='font-semibold mb-2'),
            Div(id='file-browser-body'),
            Div(Form(fh.Button('Close', cls='btn btn-sm'), method='dialog'), cls='modal-action'),
            cls='modal-box'),
        # DaisyUI's click-outside-to-close pattern: a full-screen backdrop form whose submit
        # (triggered by any click on it, since it has no other interactive content) just closes
        # the <dialog> via method='dialog', same as the Close button.
        Form(fh.Button('close', cls='cursor-default'), method='dialog', cls='modal-backdrop'),
        id='file-modal', cls='modal')


In [ ]:
#| export
def help_modal() -> FT:
    "Keyboard-shortcuts cheat sheet, opened by the '?' button in the top bar. Same modal/backdrop pattern as file_browser_modal()."
    command_mode = [
        ('j / ↓ , k / ↑', 'Select the next / previous cell'),
        ('a', 'Insert a new cell above the selection'),
        ('b', 'Insert a new cell below the selection'),
        ('y', 'Change the selected cell to Code'),
        ('m', 'Change the selected cell to Note'),
        ('r', 'Change the selected cell to Raw'),
        ('c', 'Copy the selected cell'),
        ('x', 'Cut the selected cell'),
        ('v', 'Paste after the selected cell'),
        ('w', 'Copy code blocks from the selected Assistant reply into new Code cells below it'),
        ('d d', 'Delete the selected cell (press d twice)'),
        ('s', 'Save the notebook'),
    ]
    editing_mode = [
        ('Shift / Ctrl / Cmd + Enter', 'Save (and run) the cell'),
        ('Ctrl / Cmd + /', 'Toggle line comments'),
        ('Ctrl / Cmd + -', 'Split the cell at the cursor'),
        ('Esc', 'Cancel editing, discarding unsaved changes'),
    ]
    def section(title, pairs):
        # kbd-sm rendered visibly smaller than the description text next to it -- drop the size
        # modifier (DaisyUI's default kbd size) and match font-size explicitly to the description.
        return [Div(title, cls='text-xs font-semibold opacity-60 mt-3 mb-1 first:mt-0')] + [
            Div(Kbd(k, cls='kbd text-sm'), Span(v, cls='text-sm'), cls='flex items-center gap-3 py-1') for k, v in pairs]
    return Dialog(
        Div(
            Div('Keyboard shortcuts', cls='font-semibold mb-2'),
            *section('Command mode (click a cell to select it)', command_mode),
            *section('While editing a cell', editing_mode),
            Div(Form(fh.Button('Close', cls='btn btn-sm'), method='dialog'), cls='modal-action'),
            cls='modal-box'),
        Form(fh.Button('close', cls='cursor-default'), method='dialog', cls='modal-backdrop'),
        id='help-modal', cls='modal')


In [ ]:
#| export
_TOOL_SOURCE_LABELS = {'boopiter': 'boopiter', 'slmn-nbtools': 'nbtools', 'slmn-misc': 'misc', 'slmn-remote': 'remote'}  # key (matches DEFAULT_TOOL_SELECTION) -> short label shown in the Tools menu, without the 'slmn-' implementation detail

def tools_menu(icon_cls:str) -> FT:
    "Wrench-icon 'Tools' menu: hover to see checkboxes toggling which tool sources (see _TOOL_SOURCE_LABELS/nb.tool_selection) get offered to the LLM on Prompt-cell runs -- see get_tool_list(). DaisyUI's dropdown-hover opens/closes purely via CSS on mouseenter/mouseleave, so no close button is needed."
    rows = [
        Li(Label(
            Input(type='checkbox', checked=nb.tool_selection.get(key, False), cls='checkbox checkbox-sm',
                  hx_post=toggle_tool_source.to(key=key), hx_swap='none'),
            Span(label), cls='flex items-center gap-2 px-2 py-1 cursor-pointer'))
        for key, label in _TOOL_SOURCE_LABELS.items()
    ]
    return Div(
        # inline-flex items-center justify-center: DaisyUI's .tooltip class overrides .btn's own
        # display:inline-flex (to display:inline-block), which silently breaks the icon's flexbox
        # centering inside the circle -- re-asserting it here (Tailwind's utility layer beats
        # DaisyUI's component layer) is what actually keeps the icon centered, not left-shifted.
        fh.Button(Icon('wrench-screwdriver', cls=icon_cls), data_tip='Tool selection',
                  cls='btn btn-ghost btn-circle btn-sm tooltip tooltip-bottom inline-flex items-center justify-center'),
        Ul(Li(Div('Tools', cls='menu-title')), *rows, tabindex='0', cls='dropdown-content menu bg-base-200 rounded-box z-10 w-36 p-1 shadow'),
        cls='dropdown dropdown-hover dropdown-bottom')


In [ ]:
#| export
def _plugin_slug(name:str) -> str:
    "A plugin name as a DOM-id-safe slug, e.g. 'System monitor' -> 'system-monitor'."
    return re.sub(r'[^a-z0-9]+', '-', name.lower()).strip('-')

In [ ]:
#| export
def _plugin_panel(name:str) -> FT:
    "Self-polling wrapper around the named plugin's render(): re-fetches itself (outerHTML, carrying its own hx-trigger along so polling continues) roughly every 2s, but the JS trigger filter only actually issues that request while the mouse is over the plugin's dropdown -- so the panel is inert (no requests) until hovered, and stops again the moment you mouse away. Mirrors the _run_output_div/run_code_poll self-polling idiom already used for streaming code/prompt cells. Keyed by name (not list index) -- fastcore's qp() treats 0 as falsy (0 in (False,None) is True in Python) and silently drops an idx=0 query param, so an int index is a trap for the first-registered plugin."
    pl = next(p for p in PLUGINS if p.name == name)
    dom_id = f'plugin-panel-{_plugin_slug(name)}'
    return Div(pl.render(), id=dom_id,
               hx_get=plugin_panel_poll.to(name=name), hx_target=f'#{dom_id}', hx_swap='outerHTML',
               hx_trigger="every 2s [this.closest('.dropdown').matches(':hover')]")

In [ ]:
#| export
@rt
def plugin_panel_poll(name:str) -> FT:
    "Poll target for _plugin_panel -- just re-renders the same self-polling wrapper, which is what keeps the sparklines visibly scrolling while a plugin's panel is open."
    return _plugin_panel(name)

In [ ]:
#| export
def _plugin_button(pl:Plugin) -> FT:
    "The top-bar button for a registered plugin. For trigger=='hover' (the only kind wired up so far), same pure-CSS dropdown-hover shell as tools_menu()/brain_menu() for showing/hiding the panel, but the panel content itself is the self-polling _plugin_panel() so it keeps refreshing (e.g. sparklines scrolling) for as long as it's actually visible. `left:50%; transform:translateX(-50%)` centers the popup on the button (see brain_menu()'s identical trick) rather than right-anchoring it with dropdown-end, which breaks down in a narrow browser window."
    icon = fh.Button(_svg_icon(pl.icon, cls=_TOPBAR_ICON_CLS_LG), data_tip=pl.name,
                     cls='btn btn-ghost btn-circle btn-sm tooltip tooltip-bottom inline-flex items-center justify-center')
    return Div(icon, Ul(Li(_plugin_panel(pl.name)), tabindex='0', style='left:50%; transform:translateX(-50%);',
                        cls='dropdown-content menu bg-base-200 rounded-box z-10 shadow'),
               cls='dropdown dropdown-hover dropdown-bottom')

In [ ]:
#| export
# 19px landed between size-4 (16px, too small next to the moon/sun) and size-5 (20px, ended up
# looking bigger than the moon/sun -- these are stroked outline icons vs. theme_swap()'s filled
# ones, so the same box reads heavier); 18px still read as closer to the smaller end. Shared as a
# constant (not just a local in top_bar()) so toggle_reasoning() can re-render brain_menu() at the
# same size when it refreshes itself out-of-band.
_TOPBAR_ICON_CLS = 'size-[19px]'
_TOPBAR_ICON_CLS_LG = 'size-[21px]'  # ~10% bigger than _TOPBAR_ICON_CLS -- plugin icons/brain/help/interrupt/restart-kernel read a touch small next to their neighbors at 19px

def top_bar() -> FT:
    "The whole navbar: file menu + logo + filename on the left, model pickers + kernel/theme controls on the right."
    brand = Div(file_menu(), Img(src='/logo.png', cls='h-8 w-8 rounded-full'),
                Span('boopiter', cls='font-bold text-lg'),
                Span('/', cls='opacity-40'), fname_display(),
                cls='flex items-center gap-2')
    icon_cls = _TOPBAR_ICON_CLS
    # No flex-wrap on this inner group -- these seven stay together as one atomic unit when the
    # outer ctrls row wraps, instead of a menu "stealing" just the first icon's worth of leftover
    # space on its own line and orphaning it from its siblings.
    # inline-flex items-center justify-center on every tooltip'd button: DaisyUI's .tooltip class
    # overrides .btn's own display:inline-flex (to block/inline-block), which silently breaks the
    # icon's flexbox centering inside the circle, left-shifting it -- re-asserting flex centering
    # here (Tailwind's utility layer beats DaisyUI's component layer) is the actual fix, not icon
    # size or spacing (brain_menu()'s button looked "different" only because it has no tooltip).
    tt_cls = 'btn btn-ghost btn-circle btn-sm tooltip tooltip-bottom inline-flex items-center justify-center'
    icon_btns = Div(
        # Registered plugins (see plugins.py), reversed so the first-registered one sits nearest
        # the Tools icon and later ones stack further left.
        *[_plugin_button(pl) for pl in reversed(PLUGINS)],
        tools_menu(icon_cls),
        brain_menu(_TOPBAR_ICON_CLS_LG),
        fh.Button(Icon('question-mark-circle', cls=_TOPBAR_ICON_CLS_LG), data_tip='Keyboard shortcuts', cls=tt_cls,
                  onclick="document.getElementById('help-modal').showModal()"),
        fh.Button(Icon('x-circle', cls=_TOPBAR_ICON_CLS_LG), data_tip='Interrupt kernel', cls=tt_cls,
                  hx_post=interrupt_kernel, hx_swap='none'),
        fh.Button(Icon('arrow-path', cls=_TOPBAR_ICON_CLS_LG), data_tip='Restart kernel', cls=tt_cls,
                  hx_post=restart_kernel, hx_swap='none'),
        fh.Button(Icon('play-circle', cls=icon_cls), data_tip='Run all code cells', cls=tt_cls,
                  hx_post=run_all, hx_swap='none'),
        theme_swap(), cls='flex items-center gap-1 shrink-0')
    # flex-wrap by default (so brand/ctrls stack on genuinely narrow/mobile viewports -- see the
    # mobile-layout fix earlier), but md:flex-nowrap forces a single row at tablet width and up, so
    # adding one more icon button doesn't tip ordinary desktop widths into wrapping unnecessarily.
    # ml-auto (not just the parent navbar's justify-between) is what actually keeps ctrls pinned to
    # the right once it wraps onto its own line -- justify-between only spaces multiple items *on
    # the same line* apart; once brand/ctrls each get a line to themselves, justify-between has
    # nothing to space ctrls apart from, so it silently collapses to flex-start (left) without this.
    ctrls = Div(icon_btns, cls='flex items-center gap-1 flex-wrap md:flex-nowrap justify-end ml-auto')
    return Div(brand, ctrls, file_browser_modal(), help_modal(), context_menu(),
               cls='navbar bg-base-200 shadow px-4 flex flex-wrap md:flex-nowrap items-center justify-between gap-y-1 shrink-0')


In [ ]:
#| export
@rt('/_boopiter_ping')
def boopiter_ping() -> str:
    "Identity check so `boopiter launch` can tell a live boopiter instance apart from something else on the port."
    return 'boopiter'

In [ ]:
#| export
@rt('/logo.png')
def logo_png() -> FileResponse:
    "Serve the boopiter logo, used both as favicon and in the top bar."
    return FileResponse(Path(__file__).parent.parent/'images/logo.png')

In [ ]:
#| export
@rt('/tailwind.css')
def tailwind_css() -> FileResponse:
    "Serve the precompiled Tailwind CSS built by _build_tailwind() at launch."
    return FileResponse(Path(__file__).parent/'static/tailwind.css')

In [ ]:
#| export
@rt
def index() -> tuple:
    "The full page: title, top bar, the notebook+composer, and the save-toast slot."
    return (Title('boopiter'),
            Div(top_bar(),
                # No hard max-width -- margins scale with viewport (up to a cap) instead of the
                # content column stopping short and leaving big dead space on a widened window,
                # which is exactly what you want more of when a long code line runs off-screen.
                # The right margin is nudged in by 4px (border-l-4's width) to compensate: cells'
                # colored left border sits flush against the left margin, and that hard color
                # edge visually reads as *the* boundary, making the left margin look narrower
                # than the right even though both insets are numerically equal.
                Div(Div(render_app(), cls='py-4',
                        style='padding-left: clamp(0.5rem, 4vw, 100px); '
                              'padding-right: calc(clamp(0.5rem, 4vw, 100px) - 4px)'),
                    cls='flex-1 overflow-y-auto'),
                Div(id='save-toast', cls='toast toast-top toast-end z-50'),
                cls='h-screen flex flex-col'))

In [ ]:
#| export
def _toast(msg:str, ok:bool=True) -> FT:
    "A little 'Saved' (or error) notice, out-of-band-swapped into #save-toast, that clears itself after ~1.8s."
    return Div(
        Div(msg, cls=f"alert {'alert-success' if ok else 'alert-error'} shadow-lg text-sm py-2 px-4"),
        Script("setTimeout(function(){ var t=document.getElementById('save-toast'); if(t) t.innerHTML=''; }, 1800)"),
        id='save-toast', cls='toast toast-top toast-end z-50', hx_swap_oob='true')

In [ ]:
#| export
@rt
def save_now() -> FT:
    "Save the notebook to disk and show a toast confirming success (or failure)."
    try:
        p = save_notebook()
        return _toast(f'Saved {p.name}')
    except Exception as e:
        return _toast(f'Save failed: {e}', ok=False)

In [ ]:
#| export
def _safe_dir(path:str|None) -> Path:
    "Resolve `path` (relative to BROWSE_ROOT) and clamp it back to BROWSE_ROOT if it tries to escape (e.g. via '..')."
    cur = (BROWSE_ROOT/(path or '')).resolve()
    if cur != BROWSE_ROOT and BROWSE_ROOT not in cur.parents: cur = BROWSE_ROOT
    if not cur.is_dir(): cur = BROWSE_ROOT
    return cur

In [ ]:
#| export
@rt
def browse(path:str|None=None) -> FT:
    "Jupyter-tree-style directory listing for the file-browser modal, rooted at BROWSE_ROOT."
    cur = _safe_dir(path)
    rel = cur.relative_to(BROWSE_ROOT)
    entries = sorted(cur.iterdir(), key=lambda p: (p.is_file(), p.name.lower()))
    # Reuse DaisyUI's own `menu` component (same as file_menu()'s hamburger dropdown, which also
    # skips the `-sm` size modifier) so the highlight color and font size both match exactly.
    row_cls = 'flex items-center gap-2 py-1 px-2'
    rows = []
    if cur != BROWSE_ROOT:
        up = '' if rel.parent == Path('.') else str(rel.parent)
        rows.append(Li(fh.A(Icon('folder'), ' ..', hx_get=browse.to(path=up),
                             hx_target='#file-browser-body', cls=row_cls)))
    for p in entries:
        if p.name.startswith('.'): continue
        relp = str(p.relative_to(BROWSE_ROOT))
        if p.is_dir():
            rows.append(Li(fh.A(Icon('folder'), ' ' + p.name, hx_get=browse.to(path=relp),
                                 hx_target='#file-browser-body', cls=row_cls)))
        elif p.suffix == '.ipynb':
            rows.append(Li(fh.A(Icon('document-text'), ' ' + p.name, href=open_file.to(path=relp), cls=row_cls)))
        else:
            rows.append(Li(Div(Icon('document-text'), Span(p.name), cls=row_cls + ' opacity-40'),
                            cls='pointer-events-none'))
    return Div(Div('/' if str(rel) == '.' else f'/{rel}', cls='font-mono text-xs opacity-60 mb-2'),
               Ul(*rows, cls='menu p-0'), id='file-browser-body')


In [ ]:
#| export
@rt
def open_file(path:str) -> RedirectResponse:
    "Open a notebook found by the file browser. Redirects to / afterward so the address bar (and thus a later page reload) doesn't stay pinned to this action -- reloading /open_file?path=... would silently re-run the load and discard any unsaved edits made since."
    target = _safe_dir(Path(path).parent)/Path(path).name
    try: load_notebook(target.relative_to(BROWSE_ROOT))  # relative, to match the CLI's nb.name style
    except Exception: pass
    return RedirectResponse('/', status_code=303)

In [ ]:
#| export
@rt
def new_notebook() -> RedirectResponse:
    "Discard the current notebook and start a blank one. Redirects to / afterward -- same reasoning as open_file()."
    nb.reset()
    return RedirectResponse('/', status_code=303)

In [ ]:
#| export
@rt
def restart_server() -> str:
    "nbdev-export the notebooks as a real (blocking) subprocess -- so we get a genuine exit code instead of guessing a delay -- then clear __pycache__ (WSL/Windows-mounted filesystems can have coarse mtime resolution, which can trick Python into serving stale cached bytecode right after a fast save-then-restart) and exec a fresh copy of this process (same PID, same port). Aborts (leaves the current process running) if export fails. Does NOT save the notebook first; Save manually beforehand if you want to keep unsaved edits."
    def _do_restart() -> None:
        "Runs in a background thread: exports, clears the bytecode cache, and execs a fresh process."
        time.sleep(0.3)  # let the HTTP response reach the browser before we touch anything disruptive
        exe = Path(sys.executable).parent/'nbdev-export'
        exe = str(exe) if exe.exists() else 'nbdev-export'
        # Run from the boopiter repo root, not the process's launch cwd -- if boopiter was
        # started from elsewhere (e.g. a sibling repo, to jump between projects), nbdev-export
        # would otherwise run rootless and silently export nothing, so a restart would quietly
        # keep serving the OLD code with no error surfaced anywhere. cwd= only affects this
        # subprocess, not our own process's cwd, so the exec below still resolves sys.argv[0]
        # and the notebook path relative to wherever we were actually launched from.
        repo_root = Path(__file__).parent.parent
        result = subprocess.run([exe], capture_output=True, text=True, cwd=repo_root)
        if result.returncode != 0:
            print(f'nbdev-export failed ({result.returncode}), aborting restart:\n{result.stderr}', file=sys.stderr, flush=True)
            return
        cache_dir = Path(__file__).parent/'__pycache__'
        if cache_dir.exists(): shutil.rmtree(cache_dir, ignore_errors=True)
        port = os.environ.get('BOOPITER_PORT', '8000')
        args = [sys.executable, sys.argv[0]]
        if nb.name != 'untitled': args.append(f'{nb.name}.ipynb')
        args += ['--port', port]
        os.execv(sys.executable, args)
    threading.Thread(target=_do_restart, daemon=True).start()
    return ''

In [ ]:
#| export
@rt
def shutdown_server() -> str:
    "Cleanly exit boopiter: send this process SIGTERM, the same graceful-shutdown signal Ctrl-C sends, which uvicorn catches to drain in-flight requests and exit 0. Does NOT save the notebook first -- Save manually beforehand if you want to keep unsaved edits."
    def _do_shutdown() -> None:
        "Runs in a background thread: waits for the HTTP response to reach the browser, then signals this same process to exit."
        time.sleep(0.3)  # let the HTTP response reach the browser before we touch anything disruptive
        os.kill(os.getpid(), signal.SIGTERM)
    threading.Thread(target=_do_shutdown, daemon=True).start()
    return ''

In [ ]:
#| export
@rt
def download() -> FileResponse:
    "Save the notebook, then send it to the browser as a file download."
    p = save_notebook()
    return FileResponse(str(p), filename=p.name)

In [ ]:
#| export
@rt
def rename(name:str) -> FT:
    "Rename and persist the notebook to `{new_name}.ipynb` in the server's cwd."
    nb.name = name.strip() or nb.name
    save_notebook()
    return fname_display()

In [ ]:
#| export
@rt
def restart_kernel() -> str:
    "Reset the shared IPython shell's namespace, clearing all user-defined variables/functions, then re-pushing nb/add_tool/the current tool selection -- _shell.reset() wipes those along with everything else."
    _shell.reset()          # clear kernel namespace
    _shell.push({'nb': nb, 'add_tool': add_tool})
    _push_tools()
    return ''

In [ ]:
#| export
@rt
def run_all() -> FT:
    "Run every code cell top-to-bottom, one at a time -- exactly like clicking play on each in turn. Each cell stamps its own timestamp, streams its own output, and scrolls into view as it starts; the sequence stops at the first cell that errors (see run_code_poll's chaining). Prompt/Note/Raw/Assistant cells are skipped."
    global _run_all_queue, _run_all_current
    ids = [c.id for c in nb.cells if c.ctype == 'code']
    if not ids: return ''
    _run_all_queue = ids[1:]
    first = nb.get(ids[0])
    _run_all_current = first.id
    return _oob(f'outerHTML:#cell-{first.id}', run_code_cell(first, scroll=True))

In [ ]:
#| export
@rt
def interrupt_kernel() -> str:
    "Best-effort interrupt: inject a KeyboardInterrupt into whichever background thread(s) are currently running -- a code cell (_run_state) and/or a streaming Prompt reply (_prompt_state), since those are independent slots and could both be active. Uses CPython's PyThreadState_SetAsyncExc, the same low-level trick real kernels use for SIGINT-based interrupts. It fires at the interrupted thread's next bytecode boundary, so it reliably breaks ordinary Python loops (e.g. a tqdm-wrapped for-loop, or the token-by-token loop in _run_prompt_bg) but can't preempt a single blocking C call already in flight."
    import ctypes
    for st in (_run_state, _prompt_state):
        if st is not None and st.thread is not None and st.thread.is_alive():
            tid = ctypes.c_long(st.thread.ident)
            res = ctypes.pythonapi.PyThreadState_SetAsyncExc(tid, ctypes.py_object(KeyboardInterrupt))
            if res > 1:  # affected more than one thread -- back it out
                ctypes.pythonapi.PyThreadState_SetAsyncExc(tid, None)
    return ''


In [ ]:
#| export
@rt
def toggle_tool_source(key:str) -> str:
    "Toggle one entry in nb.tool_selection (the wrench-icon Tools menu -- see top_bar()). The checkbox already flips itself natively in the browser; this just keeps the server-side selection in sync, so no DOM update is needed in response."
    nb.tool_selection[key] = not nb.tool_selection.get(key, False)
    return ''

In [ ]:
#| export
@rt
def set_type(t:str) -> FT:
    "Change the composer's current cell type (what a new cell becomes when you hit Boop)."
    if t in CTYPES: nb.compose_type = t
    return composer()

In [ ]:
#| export
def add_cell(t:str, source:str) -> list[Cell]:
    "Create a cell of type `t`: run it if code, just create it if prompt (its Assistant reply streams in asynchronously -- see _start_prompt_run/pending_prompt_cell). Returns the new cell(s)."
    if t == 'code':
        return [nb.add('code', source, output=run_code(source))]
    elif t == 'prompt':
        return [nb.add('prompt', source)]
    else:
        return [nb.add(t, source)]


In [ ]:
#| export
@rt
def submit_cell(source:str) -> tuple:
    "Append only the new cell(s) to #notebook and reset the composer out-of-band, so untouched cells' editors are never re-created. A just-submitted Prompt gets a 'Tricky...' placeholder that streams its own reply."
    new = add_cell(nb.compose_type, source) if source.strip() else []
    pending = []
    if new and nb.compose_type == 'prompt':
        _start_prompt_run(new[-1].id)
        pending = [pending_prompt_cell(new[-1].id)]
    return *[render_cell(c) for c in new], *pending, composer(oob=True)


In [ ]:
#| export
@rt
def split(source:str, pos:int) -> tuple:
    "Split the composer at the caret: head becomes a cell, tail stays in the composer."
    head, tail = source[:pos], source[pos:]
    new = add_cell(nb.compose_type, head) if head.strip() else []
    pending = []
    if new and nb.compose_type == 'prompt':
        _start_prompt_run(new[-1].id)
        pending = [pending_prompt_cell(new[-1].id)]
    return *[render_cell(c) for c in new], *pending, composer(draft=tail, oob=True)


In [ ]:
#| export
@rt
def split_cell(id:int, pos:int, source:str|None=None) -> FT|tuple:
    "Split an existing cell at caret position `pos`: it keeps the text before the cursor; a new cell of the same type (and, for code, the same export flag) is inserted right after it with the text after the cursor. `source` is the editor's live buffer, sent along by boopSplitCell() because `pos` was measured against *it*, not against the server's last-saved copy -- splitting a cell you'd typed into but not yet saved used to slice the stale copy at an offset that indexed the new one. Neither half is (re-)executed, and any existing output is cleared -- it was produced by the *whole* original code, so it doesn't correctly belong to either fragment alone (e.g. a plot statement can end up in either half after the split, stranding old output next to code that didn't produce it). Blank lines right at the split point (e.g. the PEP8 spacer between two functions) are trimmed off the boundary -- otherwise the new cell would start with an ugly-looking leading blank line."
    c = nb.get(id)
    if not c: return ''
    if source is not None: c.source = source  # adopt the live buffer first, so `pos` and the text it was measured in agree
    head, tail = c.source[:pos], c.source[pos:]
    c.source, c.output = head.rstrip('\n'), None
    i = nb.index(c.id)
    new = nb.insert_at(i+1, c.ctype, tail.lstrip('\n'), export=c.export)
    nb.selected = new.id
    return render_cell(c), _oob(f'afterend:#cell-{c.id}', render_cell(new))


In [ ]:
#| export
@rt
def run_cell(id:int) -> FT:
    "Run this cell (Code: kick off background execution and return a placeholder that streams its progress; Prompt: kick off the LLM call and return a placeholder that streams the reply in), so the caller can target just this cell instead of the whole notebook."
    c = nb.get(id)
    if c and c.ctype == 'code':
        return run_code_cell(c)
    elif c and c.ctype == 'prompt':
        _start_prompt_run(id)
        return pending_prompt_cell(id)
    return render_nb()


In [ ]:
#| export
@rt
def toggle_vis(id:int) -> str:
    "Toggle whether this cell is visible to the LLM (shown/hidden in llm_context). Called via a plain fetch(), not htmx -- boopToggleVis() (edit.js) already updated the eye icon and cell dimming instantly and optimistically on the client; this just persists that same flip server-side, no HTML response needed."
    c = nb.get(id)
    if c: c.visible = not c.visible
    return ''


In [ ]:
#| export
@rt
def toggle_export(id:int) -> str:
    "Toggle a code cell's '#| export' flag (the bookmark icon in cell_toolbar). Called via a plain fetch(), not htmx -- boopToggleExport() (edit.js) already updated the icon's look instantly and optimistically on the client; this just persists that same flip server-side, no HTML response needed."
    c = nb.get(id)
    if c and c.ctype == 'code':
        c.export = not c.export
    return ''

In [ ]:
#| export
@rt
def del_cell(id:int) -> tuple:
    "Delete this cell (or its Prompt+Assistant pair). Removes just the affected DOM node(s) out-of-band instead of re-rendering the whole notebook."
    rng = nb.pair_range(id)
    if rng is None: return ('',)
    ids = [nb.cells[j].id for j in range(rng[0], rng[1]+1)]
    nb.remove(id)
    return tuple(Div(id=f'cell-{rid}', hx_swap_oob='delete') for rid in ids)


In [ ]:
#| export
@rt
def move_cell(id:int, delta:int) -> tuple:
    "Move this cell (or its Prompt+Assistant pair) up (delta=-1) or down (delta=1). Rather than re-rendering the whole notebook, this deletes the moved block's DOM node(s) out-of-band and reinserts them next to whichever cell/pair they swapped with -- the rest of the notebook's DOM (and the page's scroll position) is never touched."
    rng = nb.pair_range(id)
    if rng is None: return ('',)
    lo, hi = rng
    if delta < 0:
        if lo == 0: return ('',)
        nlo, _ = nb.pair_range(nb.cells[lo-1].id)
        anchor_id, swap = nb.cells[nlo].id, 'beforebegin'
    elif delta > 0:
        if hi >= len(nb.cells) - 1: return ('',)
        _, nhi = nb.pair_range(nb.cells[hi+1].id)
        anchor_id, swap = nb.cells[nhi].id, 'afterend'
    else:
        return ('',)
    block_ids = [nb.cells[j].id for j in range(lo, hi+1)]
    nb.move(id, delta)
    deletes = [Div(id=f'cell-{bid}', hx_swap_oob='delete') for bid in block_ids]
    order = reversed(block_ids) if swap == 'afterend' else block_ids  # afterend inserts stack in reverse of iteration order
    inserts = [_oob(f'{swap}:#cell-{anchor_id}', render_cell(nb.get(bid))) for bid in order]
    return (*deletes, *inserts)


In [ ]:
#| export
# --- command-mode (hotkey) routes ---
@rt
def select(id:int) -> FT|tuple|str:
    "Select this cell (highlights it and anchors j/k navigation). Updates just the newly- and previously-selected cells, not the whole notebook."
    old, c = nb.selected, nb.get(id)
    if c is None: return ''
    nb.selected = id
    old_c = nb.get(old) if (old is not None and old != id) else None
    return (render_cell(c), render_cell(old_c, oob=True)) if old_c is not None else render_cell(c)


In [ ]:
#| export
@rt
def select_delta(delta:int) -> tuple:
    "Move the selection up (delta=-1) or down (delta=1) -- the j/k hotkeys. Triggered by a global hotkey (not a specific cell's button), so the response carries out-of-band updates for whichever cells actually changed rather than a full re-render."
    old = nb.selected
    if nb.cells:
        i = nb.sel_index()
        i = (0 if delta > 0 else len(nb.cells)-1) if i is None else min(max(i+delta, 0), len(nb.cells)-1)
        nb.selected = nb.cells[i].id
    changed = {old, nb.selected} - {None}
    parts = [render_cell(c, oob=True) for cid in changed if (c := nb.get(cid)) is not None]
    return tuple(parts) if parts else ('',)


In [ ]:
#| export
@rt
def insert(where:str) -> tuple:
    "Insert a new blank cell above or below the current selection -- the a/b hotkeys. The new cell's type matches the selected cell's own type (falling back to the composer's current type if nothing's selected, or if the selection is an Assistant cell, which isn't directly authorable). Inserts the new cell out-of-band next to its anchor, and refreshes the previously-selected cell (to drop its highlight ring) rather than re-rendering the whole notebook."
    old_sel = nb.selected
    i = nb.sel_index()
    if i is None:
        new = nb.insert_at(len(nb.cells), nb.compose_type, '')
        nb.selected = new.id
        return (_oob('beforeend:#notebook', render_cell(new)),)
    anchor_id = nb.cells[i].id
    ctype = nb.cells[i].ctype if nb.cells[i].ctype in CTYPES else nb.compose_type
    pos = i if where == 'above' else i+1
    new = nb.insert_at(pos, ctype, '')
    nb.selected = new.id
    swap = 'beforebegin' if where == 'above' else 'afterend'
    parts = [_oob(f'{swap}:#cell-{anchor_id}', render_cell(new))]
    old_c = nb.get(old_sel) if old_sel != nb.selected else None
    if old_c is not None: parts.append(render_cell(old_c, oob=True))
    return tuple(parts)


In [ ]:
#| export
@rt
def pull_code_blocks() -> tuple:
    "The 'w' hotkey: copy every fenced code block out of the selected Assistant reply (or its paired Prompt) into new Code cells directly below it, in order -- a low-friction way to try running code an LLM suggested without retyping it. No-op if nothing's selected or there's no Assistant reply with code fences."
    old_sel = nb.selected
    if old_sel is None: return ()
    c = nb.get(old_sel)
    if c is not None and c.ctype == 'prompt':
        i = nb.index(c.id)
        nxt = nb.cells[i+1] if i+1 < len(nb.cells) else None
        c = nxt if nxt and nxt.ctype == 'assistant' else None
    if c is None or c.ctype != 'assistant': return ()
    blocks = re.findall(r'```[^\n]*\n(.*?)```', c.source, re.DOTALL)
    if not blocks: return ()
    pos = nb.index(c.id) + 1
    anchor_id = c.id
    parts, first_id = [], None
    for block in blocks:
        new = nb.insert_at(pos, 'code', block.rstrip('\n'))
        pos += 1
        parts.append(_oob(f'afterend:#cell-{anchor_id}', render_cell(new)))
        anchor_id = new.id
        if first_id is None: first_id = new.id
    nb.selected = first_id
    old_c = nb.get(old_sel) if old_sel != nb.selected else None
    if old_c is not None: parts.append(render_cell(old_c, oob=True))
    return tuple(parts)

In [ ]:
#| export
@rt
def del_selected() -> tuple:
    "Delete the currently-selected cell (or its Prompt+Assistant pair) -- the d-d hotkey."
    if nb.selected is None: return ('',)
    i = nb.sel_index()
    rng = nb.pair_range(nb.selected)
    ids = [nb.cells[j].id for j in range(rng[0], rng[1]+1)] if rng else [nb.selected]
    nb.remove(nb.selected)
    nb.selected = nb.cells[min(i, len(nb.cells)-1)].id if nb.cells else None
    parts = [Div(id=f'cell-{rid}', hx_swap_oob='delete') for rid in ids]
    new_c = nb.get(nb.selected) if nb.selected is not None else None
    if new_c is not None: parts.append(render_cell(new_c, oob=True))
    return tuple(parts)


In [ ]:
#| export
@rt
def cut_selected() -> tuple:
    "Cut the currently-selected cell (or its pair) to the clipboard -- the x hotkey."
    if nb.selected is None: return ('',)
    rng = nb.pair_range(nb.selected)
    ids = [nb.cells[j].id for j in range(rng[0], rng[1]+1)] if rng else [nb.selected]
    nb.cut_range(nb.selected)  # also updates nb.selected
    parts = [Div(id=f'cell-{rid}', hx_swap_oob='delete') for rid in ids]
    new_c = nb.get(nb.selected) if nb.selected is not None else None
    if new_c is not None: parts.append(render_cell(new_c, oob=True))
    return tuple(parts)


In [ ]:
#| export
@rt
def copy_selected() -> str:
    "Copy the currently-selected cell (or its pair) to the clipboard -- the c hotkey."
    if nb.selected is not None: nb.copy_range(nb.selected)
    return ''

In [ ]:
#| export
@rt
def paste_selected() -> tuple:
    "Paste the clipboard after the currently-selected cell -- the v hotkey. Inserts the pasted cell(s) out-of-band, right after their anchor (or at the end, if nothing was selected), instead of re-rendering the whole notebook."
    old_sel = nb.selected
    i = nb.sel_index()
    anchor_id = nb.cells[i].id if i is not None else None
    pasted = nb.paste_after(nb.selected)  # also updates nb.selected
    if not pasted: return ('',)
    parts = []
    if anchor_id is None:
        for c in pasted: parts.append(_oob('beforeend:#notebook', render_cell(c)))
    else:
        prev = anchor_id
        for c in pasted:
            parts.append(_oob(f'afterend:#cell-{prev}', render_cell(c)))
            prev = c.id
    old_c = nb.get(old_sel) if old_sel != nb.selected else None
    if old_c is not None: parts.append(render_cell(old_c, oob=True))
    return tuple(parts)


In [ ]:
#| export
@rt
def settype_selected(t:str) -> FT|str:
    "Change the currently-selected cell's type -- the m/y/r hotkeys."
    c = nb.get(nb.selected) if nb.selected is not None else None
    if not (c and t in CTYPES): return ''
    c.ctype = t
    return render_cell(c, oob=True)


In [ ]:
#| export
@rt
def set_ctype(id:int, t:str) -> FT:
    "Change one cell's type in place; returns just that cell so the rest of the notebook is untouched."
    c = nb.get(id)
    if c and t in CTYPES and t != c.ctype:
        c.ctype = t
        c.output = None  # stale output no longer meaningful under the new type
    return render_cell(c) if c else render_nb()

In [ ]:
#| export
# --- inline editing ---
@rt
def edit_cell(id:int) -> FT:
    "Switch this cell into its live editor (CodeMirror for code, a plain textarea otherwise)."
    c = nb.get(id)
    if not c: return render_nb()
    nb.selected = id
    return render_cell_edit(c)

In [ ]:
#| export
@rt
def view_cell(id:int) -> FT:
    "Switch this cell back to its static (non-editing) view -- used by the Cancel button and the Escape hotkey (see boopCancelEdit() in edit.js). Also clears nb.selected if it's this cell -- otherwise a code cell (whose editor is shown exactly while c.id == nb.selected, not via a separate edit flag like note/prompt/raw) would just redraw right back into edit mode."
    c = nb.get(id)
    if c and nb.selected == id: nb.selected = None
    return render_cell(c) if c else render_nb()

In [ ]:
#| export
@rt
def save_cell(id:int, source:str) -> FT|tuple:
    "Commit an edited cell's source (running it if it's code, or re-prompting the LLM if it's a prompt). Targets just this cell (and, for prompts, the paired Assistant cell) rather than the whole notebook, so editing a cell deep in a long notebook doesn't blow away scroll position."
    c = nb.get(id)
    if not c: return render_nb()
    c.source = source
    if c.ctype == 'code':
        return run_code_cell(c)
    elif c.ctype == 'prompt':
        _start_prompt_run(id)
        i = nb.index(c.id)
        nxt = nb.cells[i+1] if i+1 < len(nb.cells) else None
        if nxt is not None and nxt.ctype == 'assistant':
            # 'true'/'outerHTML' OOB swaps keep the tagged element itself, so pending_prompt_cell can carry the directive directly.
            pend = pending_prompt_cell(id, oob_swap=f'outerHTML:#cell-{nxt.id}')
        else:
            # Positional OOB swaps (afterend/beforebegin/beforeend) insert only the tagged element's
            # *children* -- so pending_prompt_cell must be wrapped, not itself carry the hx-swap-oob attribute,
            # or its own id/hx-trigger="load" would be discarded on insertion. See _oob().
            pend = _oob(f'afterend:#cell-{c.id}', pending_prompt_cell(id))
        return render_cell(c), pend
    return render_cell(c)


In [ ]:
#| export
@rt
def sync_cell(id:int, source:str) -> str:
    "Update a cell's source WITHOUT executing it -- used by Save to flush any editor content that was never explicitly run (Shift+Enter), matching Jupyter's WYSIWYG save behavior."
    c = nb.get(id)
    if c: c.source = source
    return ''

In [ ]:
# Run-All queue chaining, tested without any real background thread/timing (see run_all/run_code_poll):
# construct an already-finished _RunState by hand and drive run_code_poll() directly, so the test is
# fully deterministic. Covers the two behaviors run_all()'s docstring promises: advance on success,
# stop-on-error abandons the rest of the batch untouched.
nb.reset()
c1 = nb.add('code', 'a'); c2 = nb.add('code', 'b'); c3 = nb.add('code', 'c')

def _finish(cell_id, blocks):
    "Simulate a background code run finishing -- same shape _run_code_bg produces -- without a real thread."
    global _run_state
    st = _RunState(cell_id)
    st.blocks, st.done = blocks, True
    _run_state = st

# Seed the same Run-All state run_all() would for this 3-cell batch (skipping the real thread it
# would normally start for c1 -- the chaining logic under test lives entirely in run_code_poll()).
_run_all_queue, _run_all_current = [c2.id, c3.id], c1.id

_finish(c1.id, [{'type':'stream', 'mime':None, 'data':'ok'}])
run_code_poll(c1.id, poll_n=1)
assert _run_all_current == c2.id, "success should advance Run All to the next queued cell"
assert _run_all_queue == [c3.id], "the advanced-past cell should be popped off the queue"
assert c1.output == [{'type':'stream', 'mime':None, 'data':'ok'}], "the finished cell's output should be recorded"

_finish(c2.id, [{'type':'error', 'mime':None, 'data':'boom'}])
run_code_poll(c2.id, poll_n=1)
assert _run_all_queue == [] and _run_all_current is None, "an error should abandon the rest of the batch"
assert c3.output is None, "c3 must never have been touched once the batch stopped"

nb.reset()
print('Run-All queue chaining verified: advances on success, stops on error')

In [ ]:
# A cell can be deleted while Run All is mid-batch (e.g. the user deletes a not-yet-reached cell
# while an earlier one is still running) -- run_code_poll's chaining loop should skip straight past
# it rather than choke on a missing id.
nb.reset()
c1 = nb.add('code', 'a'); c2 = nb.add('code', 'b'); c3 = nb.add('code', 'c')
_run_all_queue, _run_all_current = [c2.id, c3.id], c1.id
nb.remove(c2.id)  # deleted mid-run, while c1 is still "in flight"

_finish(c1.id, [{'type':'stream', 'mime':None, 'data':'ok'}])
run_code_poll(c1.id, poll_n=1)
assert _run_all_current == c3.id, "the deleted cell should be skipped, landing on c3 instead"
assert _run_all_queue == [], "nothing should be left queued after skipping straight to the last cell"

nb.reset()
print('Run-All queue skips a cell deleted mid-run')

## Run it

In a notebook, start the server and preview inline. Click a cell's header to
select it, then use command-mode keys: `A`/`B` insert above/below, `D D` delete,
`J`/`K` (or arrows) move selection, `M`/`Y`/`R` change type. In the composer,
`Cmd/Ctrl+/` toggles comments on the selected lines and `Cmd/Ctrl+Shift+-` splits
at the caret. On WSL see the lesson's port notes for reaching it from Windows.

In [ ]:
#| eval: false
srv = JupyUvi(app)
p(index())

NameError: name 'app' is not defined

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <ipython-input-1-db8d6d5e6a19>:2
      1 #| eval: false
----> 2 srv = JupyUvi(app)
      3 p(index())

NameError: name 'app' is not defined


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()